# SleepingAtTheTop
Open-source pipeline for ResMed AirSense 11 SD-card EDF files: custom event detection, therapy indices, and HTML reports.

**Not a medical device.** Results are for personal research and may differ from OSCAR or a clinical sleep study.

**Pipeline stages in this notebook:**
1. Configuration & imports
2. Single-night session read + sanity checks
3. BRP — flow waveform inspection
4. PLD — per-night trend signals
5. SA2 — SpO2 & pulse
6. EVE — ResMed scored events
7. STR — longitudinal summary (all nights)
8. Longitudinal plots
9. Session summary table
10. Event detection (single night)
11. Indices & CTQI (single night)
12. Batch: event detection + indices (all nights)
13. Settings changes
14. Pulse-flow correlation
15. Generate reports

---
## 1. Configuration & Imports

In [ ]:
import tkinter as tk
from tkinter import filedialog, simpledialog
from pathlib import Path
import re as _re

# Hide the blank root window that tkinter creates
_root = tk.Tk()
_root.withdraw()
_root.attributes("-topmost", True)

# ── 1. Select the top-level SD card folder ──────────────────────────────────
# This should be the folder that contains DATALOG/ and STR.edf.
# If you point at the DATALOG folder itself, we look one level up.
print("Select the SD card root folder (contains DATALOG/ and STR.edf)...")
SD_ROOT = filedialog.askdirectory(
    title="Select SD card root (contains DATALOG and STR.edf)",
    mustexist=True,
)
if not SD_ROOT:
    raise RuntimeError("No folder selected. Please re-run this cell.")
SD_ROOT = Path(SD_ROOT)

# ── 2. Auto-discover DATALOG and STR.edf ────────────────────────────────────
# If user selected the DATALOG folder directly, go up one level
if SD_ROOT.name == "DATALOG":
    SD_ROOT = SD_ROOT.parent

DATALOG_ROOT = None
for candidate in [SD_ROOT / "DATALOG", SD_ROOT]:
    nights = sorted(p for p in candidate.iterdir()
                    if p.is_dir() and _re.match(r"\d{8}", p.name))
    if nights:
        DATALOG_ROOT = candidate
        break

if DATALOG_ROOT is None:
    raise RuntimeError(f"No YYYYMMDD night folders found under {SD_ROOT}")

STR_PATH = None
for candidate in [SD_ROOT / "STR.edf", DATALOG_ROOT.parent / "STR.edf",
                  DATALOG_ROOT / "STR.edf"]:
    if candidate.exists():
        STR_PATH = candidate
        break

if STR_PATH is None:
    print("⚠ STR.edf not found automatically. Please select it manually.")
    STR_PATH = filedialog.askopenfilename(
        title="Select STR.edf",
        filetypes=[("EDF files", "*.edf"), ("All files", "*.*")],
    )
    if not STR_PATH:
        raise RuntimeError("No STR.edf selected.")
    STR_PATH = Path(STR_PATH)

# ── 3. Discover JSON files ───────────────────────────────────────────────────
ID_JSON = None
SETTINGS_JSON = None
_json_search_dirs = [SD_ROOT, DATALOG_ROOT.parent, DATALOG_ROOT,
                     SD_ROOT / "SETTINGS", DATALOG_ROOT.parent / "SETTINGS"]
for candidate_dir in _json_search_dirs:
    if not candidate_dir.is_dir():
        continue
    ij = candidate_dir / "Identification.json"
    sj = candidate_dir / "CurrentSettings.json"
    if ij.exists() and ID_JSON is None:
        ID_JSON = ij
    if sj.exists() and SETTINGS_JSON is None:
        SETTINGS_JSON = sj

# ── 4. Pick a night to inspect ───────────────────────────────────────────────
available_nights = sorted(
    p.name for p in DATALOG_ROOT.iterdir()
    if p.is_dir() and _re.match(r"\d{8}", p.name)
)
default_night = available_nights[-1]
INSPECT_DATE = simpledialog.askstring(
    title="Select Night",
    prompt=(
        f"Enter a date to inspect (YYYYMMDD).\n"
        f"Available: {available_nights[0]} → {available_nights[-1]}\n"
        f"({len(available_nights)} nights total)"
    ),
    initialvalue=default_night,
    parent=_root,
)
if not INSPECT_DATE or INSPECT_DATE not in available_nights:
    raise RuntimeError(f"Invalid date. Available: {available_nights[0]} → {available_nights[-1]}")

_root.destroy()

print(f"SD Root : {SD_ROOT}")
print(f"DATALOG : {DATALOG_ROOT}")
print(f"STR     : {STR_PATH}")
print(f"ID JSON : {ID_JSON or 'not found'}")
print(f"Settings: {SETTINGS_JSON or 'not found'}")
print(f"Night   : {INSPECT_DATE}")
print(f"Nights  : {len(available_nights)} ({available_nights[0]} → {available_nights[-1]})")
print("\nAll paths confirmed.")

In [ ]:
import sys, platform
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

# resmed_edf.py must be in the same folder as this notebook (or on sys.path)
sys.path.insert(0, str(Path(".").resolve()))
from resmed_edf import (
    read_session, read_str, session_summary, filter_spo2_zeros
)

print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("OS     :", platform.platform())

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 3)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
# ── §1b  Device & Settings Info ───────────────────────────────────────────────
# Uses ID_JSON and SETTINGS_JSON auto-discovered in cell 1 above.
import json as _json

def _print_json(d, indent=0):
    """Recursively print all keys and values in a JSON dict."""
    pad = "  " * indent
    for k, v in d.items():
        if isinstance(v, dict):
            print(f"{pad}{k}:")
            _print_json(v, indent + 1)
        elif isinstance(v, list):
            print(f"{pad}{k}:")
            for item in v:
                if isinstance(item, dict):
                    _print_json(item, indent + 1)
                else:
                    print(f"{pad}  - {item}")
        else:
            print(f"{pad}{k:40s}: {v}")

if ID_JSON and ID_JSON.exists():
    print("─" * 60)
    print("IDENTIFICATION.JSON")
    print("─" * 60)
    _print_json(_json.loads(ID_JSON.read_text()))
else:
    print("Identification.json not found — skipped.")

if SETTINGS_JSON and SETTINGS_JSON.exists():
    print("\n" + "─" * 60)
    print("CURRENTSETTINGS.JSON")
    print("─" * 60)
    _print_json(_json.loads(SETTINGS_JSON.read_text()))
else:
    print("CurrentSettings.json not found — skipped.")

---
## 2. Single-Night Session Read

In [ ]:
session_folder = Path(DATALOG_ROOT) / INSPECT_DATE
print("Reading:", session_folder)

session = read_session(session_folder)
meta    = session["meta"]

session_start = meta['session_start']
dur_hr = meta.get('duration_hr', 0)
session_end = session_start + timedelta(hours=dur_hr)

print(f"\nSession start : {session_start}")
print(f"Session end   : {session_end}")
print(f"Duration      : {dur_hr:.2f} hrs")
print()

# Show what signals were loaded
for token in ("BRP", "PLD", "SA2"):
    data = session.get(token, {})
    if not data:
        print(f"  {token}: NOT FOUND")
        continue
    for label, arr in data.items():
        fs = meta.get(token, {}).get("fs", {}).get(label, "?")
        print(f"  {token}/{label:20s}: {len(arr):>9,} samples  @ {fs} Hz")

# Split EVE into clinical events vs annotations
_annotation_labels = {"Recording starts", "Recording ends"}
eve_all = session["EVE"]
eve_clinical = [e for e in eve_all if e["label"] not in _annotation_labels]
eve_bookkeeping = [e for e in eve_all if e["label"] in _annotation_labels]
print(f"\n  EVE events   : {len(eve_clinical)} clinical + {len(eve_bookkeeping)} annotation ({len(eve_all)} total)")
print(f"  CSL events   : {len(session['CSL'])}")

In [ ]:
# Quick summary (rounded for readability)
summary = session_summary(session)
rounded = {}
for k, v in summary.items():
    if isinstance(v, float):
        rounded[k] = round(v, 2)
    else:
        rounded[k] = v

import pprint
pprint.pprint(rounded)

---
## 3. BRP — Flow Waveform

**BRP signals:** `Flow.40ms` (L/s, 25 Hz), `Press.40ms` (cmH2O, 25 Hz)  
Positive flow = inhalation. Negative = exhalation.

In [ ]:
brp = session["BRP"]
flow   = brp["Flow.40ms"]     # L/s
press  = brp["Press.40ms"]    # cmH2O

fs_brp = meta["BRP"]["fs"]["Flow.40ms"]   # should be 25.0 Hz
t_brp  = np.arange(len(flow)) / fs_brp    # seconds from session start
t_brp_hr = t_brp / 3600                   # hours from session start

print(f"BRP sampling rate: {fs_brp} Hz")
print(f"Flow range  : {np.nanmin(flow):.3f} to {np.nanmax(flow):.3f} L/s")
print(f"Press range : {np.nanmin(press):.3f} to {np.nanmax(press):.3f} cmH2O")

In [ ]:
# ── Plot first 5 minutes of flow waveform ──────────────────────────────────
window_s = 300  # 5 minutes
n = int(window_s * fs_brp)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

axes[0].plot(t_brp[:n], flow[:n], lw=0.5, color="steelblue")
axes[0].axhline(0, color="gray", lw=0.5, ls="--")
axes[0].set_ylabel("Flow (L/s)")
axes[0].set_title(f"BRP — First {window_s//60} minutes  ({INSPECT_DATE})")

axes[1].plot(t_brp[:n], press[:n], lw=0.5, color="darkorange")
axes[1].set_ylabel("Pressure (cmH2O)")
axes[1].set_xlabel("Time from session start (seconds)")

plt.tight_layout()
plt.show()

In [ ]:
# ── Plot full-night flow overview (downsampled for speed) ──────────────────
# Downsample to 1 Hz by taking every 25th sample
step = max(1, int(fs_brp))
flow_ds = flow[::step]
t_ds    = t_brp_hr[::step]

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t_ds, flow_ds, lw=0.3, color="steelblue", alpha=0.7)
ax.axhline(0, color="gray", lw=0.5, ls="--")
ax.set_xlabel("Hours from session start")
ax.set_ylabel("Flow (L/s)")
ax.set_title("Full-night flow (downsampled to 1 Hz for display)")

# Overlay EVE events as vertical lines
colors = {
    "Obstructive Apnea": "red",
    "Central Apnea":     "purple",
    "Hypopnea":          "orange",
    "RERA":              "green",
}
session_start = meta["session_start"]
for ev in session["EVE"]:
    if ev["label"] == "Recording starts":
        continue
    hr = ev["onset_s"] / 3600
    color = colors.get(ev["label"], "gray")
    ax.axvline(hr, color=color, alpha=0.8, lw=1.2)

legend_patches = [Patch(color=c, label=l) for l, c in colors.items()]
ax.legend(handles=legend_patches, fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

---
## 4. PLD — Low-Resolution Trend Signals

**PLD signals (all at 0.5 Hz / 1 sample per 2s):**  
`MaskPress.2s`, `Press.2s`, `EprPress.2s`, `Leak.2s`, `RespRate.2s`,  
`TidVol.2s`, `MinVent.2s`, `Snore.2s`, `FlowLim.2s`

In [ ]:
pld    = session["PLD"]
fs_pld = meta["PLD"]["fs"]["Leak.2s"]   # 0.5 Hz
t_pld  = np.arange(len(pld["Leak.2s"])) / fs_pld / 3600  # hours

print("PLD sampling rate:", fs_pld, "Hz")
for sig, arr in pld.items():
    valid = arr[np.isfinite(arr)]
    print(f"  {sig:20s}: min={valid.min():.3f}  median={np.median(valid):.3f}  "
          f"p95={np.percentile(valid,95):.3f}  max={valid.max():.3f}")

In [ ]:
# ── Multi-panel PLD plot ───────────────────────────────────────────────────
pld_signals = [
    ("Press.2s",    "Pressure (cmH2O)",  "darkorange"),
    ("Leak.2s",     "Leak (L/s)",        "crimson"),
    ("RespRate.2s", "Resp Rate (bpm)",   "steelblue"),
    ("Snore.2s",    "Snore Index",       "purple"),
    ("FlowLim.2s",  "Flow Limitation",   "teal"),
]

fig, axes = plt.subplots(len(pld_signals), 1, figsize=(14, 10), sharex=True)
fig.suptitle(f"PLD — Full Night  ({INSPECT_DATE})", y=1.01)

for ax, (sig, ylabel, color) in zip(axes, pld_signals):
    ax.plot(t_pld, pld[sig], lw=0.8, color=color)
    ax.set_ylabel(ylabel, fontsize=8)
    # EVE event markers
    for ev in session["EVE"]:
        if ev["label"] == "Recording starts":
            continue
        ax.axvline(ev["onset_s"] / 3600, color="red", alpha=0.5, lw=0.8)

axes[-1].set_xlabel("Hours from session start")
plt.tight_layout()
plt.show()

---
## 5. SA2 — SpO2 & Pulse

**SA2 signals (both at 1 Hz):** `SpO2.1s` (%), `Pulse.1s` (bpm)  
⚠️ Zero values = signal loss (oximeter off finger). Filter before stats.

In [ ]:
if not session.get("SA2"):
    print("SA2 not found for this night — oximeter file missing or not recorded.")
else:
    sa2    = session["SA2"]
    fs_sa2 = meta["SA2"]["fs"]["SpO2.1s"]   # 1.0 Hz
    t_sa2  = np.arange(len(sa2["SpO2.1s"])) / fs_sa2 / 3600  # hours

    spo2  = filter_spo2_zeros(sa2["SpO2.1s"])   # zeros -> NaN
    pulse = filter_spo2_zeros(sa2["Pulse.1s"])  # zeros -> NaN

    valid_spo2  = spo2[np.isfinite(spo2)]
    valid_pulse = pulse[np.isfinite(pulse)]

    print(f"SpO2  — median: {np.median(valid_spo2):.1f}%   "
          f"min: {np.min(valid_spo2):.1f}%   "
          f"time <90%: {np.sum(valid_spo2 < 90) / fs_sa2 / 60:.1f} min")
    print(f"Pulse — median: {np.median(valid_pulse):.0f} bpm   "
          f"min: {np.min(valid_pulse):.0f}   max: {np.max(valid_pulse):.0f}")


In [ ]:
if not session.get("SA2"):
    print("SA2 not found — skipping SpO2/pulse plot.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    fig.suptitle(f"SA2 — SpO2 & Pulse  ({INSPECT_DATE})")

    axes[0].plot(t_sa2, spo2, lw=0.6, color="steelblue")
    axes[0].axhline(90, color="red", lw=0.8, ls="--", label="90% threshold")
    axes[0].axhline(95, color="orange", lw=0.8, ls="--", label="95% threshold")
    axes[0].set_ylabel("SpO2 (%)")
    axes[0].set_ylim(80, 102)
    axes[0].legend(fontsize=8)

    axes[1].plot(t_sa2, pulse, lw=0.6, color="crimson")
    axes[1].set_ylabel("Pulse (bpm)")
    axes[1].set_xlabel("Hours from session start")
    # Clamp pulse axis to valid physiological range — excludes any sentinel
    # values that escaped filtering and prevents -1 from collapsing the axis
    valid_pulse_range = pulse[np.isfinite(pulse)]
    if len(valid_pulse_range) > 0:
        axes[1].set_ylim(
            max(20,  np.nanpercentile(valid_pulse_range, 1)  - 5),
            min(200, np.nanpercentile(valid_pulse_range, 99) + 5)
        )

    # EVE markers on both panels
    for ax in axes:
        for ev in session["EVE"]:
            if ev["label"] == "Recording starts":
                continue
            ax.axvline(ev["onset_s"] / 3600, color="gray", alpha=0.5, lw=0.8)

    plt.tight_layout()
    plt.show()

---
## 6. EVE — ResMed Scored Events

In [ ]:
events = session["EVE"]
dur_hr = meta.get("duration_hr", 0)

# Build a DataFrame
df_eve = pd.DataFrame(events)
df_eve = df_eve[df_eve["label"] != "Recording starts"].copy()
df_eve["hour"] = df_eve["onset_s"] / 3600

print(df_eve[["time", "label", "duration_s", "hour"]].to_string(index=False))
print()

counts = df_eve["label"].value_counts()
total  = counts.sum()
print("Event counts:")
print(counts.to_string())
print(f"\nTotal events : {total}")
print(f"Duration     : {dur_hr:.2f} hr")
print(f"AHI estimate : {total / dur_hr:.2f} /hr  (ResMed scored)")
total_dur_s = df_eve["duration_s"].sum()
print(f"Total time in apnea: {int(total_dur_s // 60)}:{int(total_dur_s % 60):02d}")

In [ ]:
# ── Event hypnogram-style timeline ─────────────────────────────────────────
event_y = {
    "Obstructive Apnea": 3,
    "Central Apnea":     2,
    "Hypopnea":          1,
    "RERA":              0,
}
event_color = {
    "Obstructive Apnea": "red",
    "Central Apnea":     "purple",
    "Hypopnea":          "orange",
    "RERA":              "green",
}

fig, ax = plt.subplots(figsize=(14, 3))
for _, row in df_eve.iterrows():
    y     = event_y.get(row["label"], -1)
    color = event_color.get(row["label"], "gray")
    # ResMed timestamps an event at its END; hypopneas carry no duration (10 s assumed)
    dur   = max((row["duration_s"] if row["duration_s"] > 0 else 10.0) / 3600, 0.005)
    ax.barh(y, dur, left=row["hour"] - dur, height=0.6, color=color, alpha=0.8)

ax.set_yticks(list(event_y.values()))
ax.set_yticklabels(list(event_y.keys()), fontsize=9)
ax.set_xlabel("Hours from session start")
ax.set_xlim(0, dur_hr)
ax.set_title(f"ResMed Scored Events  ({INSPECT_DATE})")
plt.tight_layout()
plt.show()

---
## 7. STR — Longitudinal Summary

One row per day. Primary source for trends across all nights.

In [ ]:
print("Reading STR:", STR_PATH)
df_str = read_str(STR_PATH)

# Drop rows with no therapy (Duration == 0 or NaN = no session that day)
df_active = df_str[df_str["Duration"] > 30].copy()

# Replace SpO2 sentinel values in STR.  ResMed writes -1.0 on nights when
# the oximeter was not worn.  Zero is also invalid.  Mask both to NaN so
# they are excluded from stats and plots.
for _col in [c for c in df_active.columns if "SpO2" in c or "Pulse" in c]:
    df_active[_col] = df_active[_col].where(df_active[_col] > 0, other=float("nan"))
for _col in [c for c in df_active.columns if "Leak" in c or "Press" in c]:
    df_active[_col] = df_active[_col].clip(lower=0)

print(f"\nTotal records in STR : {len(df_str)}")
print(f"Nights with therapy  : {len(df_active)}")
print(f"Date range           : {df_active.index[0].date()} → {df_active.index[-1].date()}")
print()

key_cols = ["Duration", "AHI", "OAI", "CAI", "HI", "RIN",
            "Leak.50", "Leak.95", "SpO2.50", "MaskPress.50", "MaskPress.95"]
avail = [c for c in key_cols if c in df_active.columns]
print(df_active[avail].describe().round(2).to_string())

In [ ]:
print("Last 10 nights:")
print(df_active[avail].tail(10).to_string())

---
## 8. Longitudinal Plots

In [ ]:
# ── AHI over time ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

ax.scatter(df_active.index, df_active["AHI"], s=8, alpha=0.6, color="steelblue", label="Nightly AHI")

# 7-day rolling median
rolling = df_active["AHI"].rolling(7, min_periods=3).median()
ax.plot(df_active.index, rolling, lw=1.5, color="steelblue", label="7-day median")

# Threshold bands
ax.axhspan(0,  5,  alpha=0.06, color="green",  label="Excellent (<5)")
ax.axhspan(5,  10, alpha=0.06, color="yellow", label="Good (5–10)")
ax.axhspan(10, 20, alpha=0.06, color="orange", label="Okay (10–20)")
ax.axhspan(20, ax.get_ylim()[1] if ax.get_ylim()[1] > 20 else 30,
           alpha=0.06, color="red", label="Bad (>20)")

ax.set_ylabel("AHI (events/hr)")
ax.set_title("Apnea-Hypopnea Index — All Nights")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha="right")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# ── Multi-panel longitudinal overview ─────────────────────────────────────
panels = [
    ("AHI",         "AHI (events/hr)",   "steelblue"),
    ("Leak.50",     "Leak median (L/s)", "crimson"),
    ("MaskPress.95","Pressure 95th (cmH2O)", "darkorange"),
    ("SpO2.50",     "SpO2 median (%)",   "teal"),
    ("Duration",    "Usage (min)",       "gray"),
]
panels = [(sig, lbl, col) for sig, lbl, col in panels if sig in df_active.columns]

fig, axes = plt.subplots(len(panels), 1, figsize=(14, 12), sharex=True)
fig.suptitle("Longitudinal Summary — All Nights", fontsize=13)

for ax, (sig, ylabel, color) in zip(axes, panels):
    ax.scatter(df_active.index, df_active[sig], s=5, alpha=0.5, color=color)
    roll = df_active[sig].rolling(7, min_periods=3).median()
    ax.plot(df_active.index, roll, lw=1.5, color=color)
    ax.set_ylabel(ylabel, fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ── AHI breakdown by event type ────────────────────────────────────────────
event_cols = {"OAI": "Obstructive", "CAI": "Central", "HI": "Hypopnea", "RIN": "RERA"}
available  = {k: v for k, v in event_cols.items() if k in df_active.columns}

fig, ax = plt.subplots(figsize=(14, 4))
bottom = np.zeros(len(df_active))
palette = ["steelblue", "purple", "orange", "green"]

for (col, label), color in zip(available.items(), palette):
    vals = df_active[col].fillna(0).values
    ax.bar(df_active.index, vals, bottom=bottom, label=label,
           color=color, alpha=0.75, width=1.0)
    bottom += vals

ax.set_ylabel("Events/hr")
ax.set_title("AHI Breakdown by Event Type")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha="right")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ────────────────────────────────────────────────────
# Selected metrics: independent dimensions of sleep quality and therapy.
# Excluded: MaskPress.95 (r=0.84 with .50), Leak.50 (no variance),
#           HI (near-zero), MinVent.50 (product of RespRate × TidVol).
corr_cols = [c for c in ["AHI", "OAI", "CAI", "RIN",
                          "Leak.95", "MaskPress.50", "SpO2.50",
                          "RespRate.50", "TidVol.50",
                          "Duration"]
             if c in df_active.columns]
corr = df_active[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(corr_cols))); ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(corr_cols))); ax.set_yticklabels(corr_cols, fontsize=9)
ax.set_title("Correlation Heatmap — Nightly Metrics")
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.tight_layout()
plt.show()

---
## 9. Batch: Build Per-Night Summary Table (legacy — SKIPPED)

**This section is disabled by default.** Section 12 below supersedes it —  
same batch loop but with event detection, indices, and CTQI.  
Set `RUN_SECTION_9 = True` in the cells below if you need the legacy table.

In [ ]:
RUN_SECTION_9 = False  # Set True to run legacy batch (slow, ~30 min)

import re

if RUN_SECTION_9:
    # ── Scan all available DATALOG subfolders ─────────────────────────────────
    datalog = Path(DATALOG_ROOT)
    date_dirs = sorted(p for p in datalog.iterdir()
                       if p.is_dir() and re.match(r"\d{8}", p.name))
    print(f"Found {len(date_dirs)} session folders")
    if date_dirs:
        print(f"  Earliest: {date_dirs[0].name}")
        print(f"  Latest  : {date_dirs[-1].name}")


In [ ]:
RUN_SECTION_9 = False  # Set True to run legacy batch (slow, ~30 min)

import re
from datetime import datetime
from tqdm.auto import tqdm   # pip install tqdm  (or remove and use plain loop)

if RUN_SECTION_9:
    rows = []
    for d in tqdm(date_dirs, desc="Reading sessions"):
        try:
            s = read_session(d)
            summ = session_summary(s)
            summ["date"] = pd.Timestamp(datetime.strptime(d.name, "%Y%m%d").date())
            # Flatten event counts into columns
            for label, cnt in summ.pop("events", {}).items():
                summ[f"evt_{label.replace(' ', '_')}"] = cnt
            rows.append(summ)
        except Exception as e:
            print(f"  Skipped {d.name}: {e}")
    if not rows:
        print("No sessions were loaded. Check the errors above.")
    else:
        df_nightly = pd.DataFrame(rows).set_index("date").sort_index()
        print(f"\nBuilt nightly table: {df_nightly.shape}")
        df_nightly.head()


In [ ]:
RUN_SECTION_9 = False  # Set True to run legacy batch (slow, ~30 min)

if RUN_SECTION_9:
    # Save to CSV for reference
    out_csv = Path(".") / "nightly_summary.csv"
    df_nightly.to_csv(out_csv)
    print("Saved:", out_csv)


---
## 10. Event Detection (single night)

Run all custom event detectors on the inspected night.  
Compare results against ResMed EVE ground truth.

In [ ]:
# ── §10a  Import analytics modules ────────────────────────────────────────────
import event_detection
import indices
import settings_changes
import report

print("Modules loaded:")
print("  event_detection :", hasattr(event_detection, 'detect_all'))
print("  indices         :", hasattr(indices, 'compute_night_indices'))
print("  settings_changes:", hasattr(settings_changes, 'detect_settings_changes'))
print("  report          :", hasattr(report, 'generate_single_night'))

In [ ]:
# ── §10b  Adapt session dict for event_detection API ──────────────────────
# read_session() returns uppercase keys ("BRP", "PLD", etc.)
# event_detection.detect_all() expects lowercase with specific sub-keys.

def adapt_session(session):
    """Convert read_session() output to detect_all() input format."""
    m = session.get("meta", {})
    adapted = {"meta": {}, "brp": None, "pld": None, "sa2": None, "eve": None}

    # Duration: from BRP if available, else PLD, else 0
    dur_s = 0.0
    for tok in ("BRP", "PLD", "SA2"):
        if tok in m and "duration_s" in m[tok]:
            dur_s = m[tok]["duration_s"]
            break

    adapted["meta"] = {
        "duration_s": dur_s,
        "start_dt":   m.get("session_start"),
        "end_dt":     None,
    }
    # dur_s is recorded time. With several mask-on sessions in one night the
    # gaps between them are excluded, so take the end time from the reader.
    for tok in ("BRP", "PLD", "SA2"):
        if tok in m and m[tok].get("end_dt"):
            adapted["meta"]["end_dt"] = m[tok]["end_dt"]
            break
    if adapted["meta"]["end_dt"] is None and adapted["meta"]["start_dt"] and dur_s > 0:
        from datetime import timedelta
        adapted["meta"]["end_dt"] = adapted["meta"]["start_dt"] + timedelta(seconds=dur_s)

    # BRP
    brp = session.get("BRP")
    if brp and "Flow.40ms" in brp:
        brp_meta = m.get("BRP", {})
        adapted["brp"] = {
            "Flow.40ms":  brp["Flow.40ms"],
            "Press.40ms": brp.get("Press.40ms"),
            "fs":         brp_meta.get("fs", {}).get("Flow.40ms", 25.0),
            "start_dt":   brp_meta.get("start_dt", m.get("session_start")),
        }

    # PLD
    pld = session.get("PLD")
    if pld:
        pld_meta = m.get("PLD", {})
        fs_pld = pld_meta.get("fs", {}).get("Leak.2s", 0.5)
        adapted["pld"] = {
            "MaskPress.2s": pld.get("MaskPress.2s"),
            "Press.2s":     pld.get("Press.2s"),
            "EprPress.2s":  pld.get("EprPress.2s"),
            "Leak.2s":      pld.get("Leak.2s"),
            "RespRate.2s":  pld.get("RespRate.2s"),
            "TidVol.2s":    pld.get("TidVol.2s"),
            "MinVent.2s":   pld.get("MinVent.2s"),
            "Snore.2s":     pld.get("Snore.2s"),
            "FlowLim.2s":   pld.get("FlowLim.2s"),
            "fs":           fs_pld,
            "start_dt":     pld_meta.get("start_dt", m.get("session_start")),
        }

    # SA2
    sa2 = session.get("SA2")
    if sa2 and ("SpO2.1s" in sa2 or "Pulse.1s" in sa2):
        sa2_meta = m.get("SA2", {})
        adapted["sa2"] = {
            "SpO2.1s":  sa2.get("SpO2.1s"),
            "Pulse.1s": sa2.get("Pulse.1s"),
            "fs":       sa2_meta.get("fs", {}).get("SpO2.1s", 1.0),
            "start_dt": sa2_meta.get("start_dt", m.get("session_start")),
        }

    # EVE
    eve = session.get("EVE")
    if eve:
        adapted["eve"] = {"events": eve}

    return adapted

session_adapted = adapt_session(session)
print(f"Adapted session:  brp={'yes' if session_adapted['brp'] else 'no'},  "
      f"pld={'yes' if session_adapted['pld'] else 'no'},  "
      f"sa2={'yes' if session_adapted['sa2'] else 'no'},  "
      f"eve={'yes' if session_adapted['eve'] else 'no'}")
print(f"Duration: {session_adapted['meta']['duration_s']/60:.1f} min")
print(f"BRP fs: {session_adapted['brp']['fs']} Hz" if session_adapted['brp'] else "")

In [ ]:
# ── §10c  Run all event detectors ─────────────────────────────────────────────

det = event_detection.detect_all(session_adapted)

if det["session_too_short"]:
    print("⚠ Session too short for analysis (< 30 min)")
else:
    print(f"=== Event Detection Results for {INSPECT_DATE} ===")
    print(f"Apneas:          {len(det['events_apnea'])}")
    print(f"Hypopneas:       {len(det['events_hypopnea'])}")
    print(f"  flow only (no oximetry, not in AHI): {len(det.get('events_hypopnea_flow_only', []))}")
    print(f"Flow Limitation: {len(det['events_flow_lim'])}")
    print(f"Desaturations:   {len(det['events_desat'])}")
    print(f"Pulse Surges:    {len(det['events_pulse_surge'])}")
    print(f"Snore Events:    {len(det['events_snore'])}")
    print(f"Total custom:    {len(det['all_custom_events'])}")
    print()
    flm = det.get('fl_metrics', {})
    print(f"FL mean: {flm.get('fl_mean', float('nan')):.3f},  "
          f"FL index: {flm.get('fl_index', float('nan')):.1f}/hr,  "
          f"FL frac>0.3: {flm.get('fl_frac_elevated', 0):.1%}")
    dsm = det.get('desat_metrics', {})
    print(f"ODI-3: {dsm.get('ODI_3', float('nan')):.1f},  "
          f"ODI-4: {dsm.get('ODI_4', float('nan')):.1f},  "
          f"T90: {dsm.get('T90', float('nan')):.1f} min,  "
          f"SpO2 nadir: {dsm.get('spo2_nadir', float('nan')):.0f}%")
    psm = det.get('pulse_metrics', {})
    print(f"PCI (6bpm): {psm.get('PCI_6bpm', float('nan')):.1f},  "
          f"PCI (8bpm): {psm.get('PCI_8bpm', float('nan')):.1f}")

In [ ]:
# ── §10d  Custom vs. ResMed EVE comparison ───────────────────────────────────

eve_events = session.get("EVE", [])
comparison = None
if eve_events:
    resp_custom = det["events_apnea"] + det["events_hypopnea"]
    comparison = event_detection.compare_with_eve(resp_custom, eve_events)
    print("=== Custom vs. ResMed EVE ===")
    print(f"Custom respiratory: {comparison['custom_total']}")
    print(f"EVE respiratory:   {comparison['eve_total']}")
    print(f"Matched (±15s):    {comparison['matched']}")
    print(f"Custom only:       {comparison['custom_only']}")
    print(f"EVE only:          {comparison['eve_only']}")
    print(f"Precision: {comparison['precision']}")
    print(f"Recall:    {comparison['recall']}")
    print(f"F1:        {comparison['f1']}")
    
    # Debug: show actual event times
    print("\nCustom event times:")
    for e in resp_custom:
        print(f"  {e.get('time')}  {e['label']:30s}  onset_s={e['onset_s']:.1f}  dur={e['duration_s']:.1f}s")
    print("\nEVE event times:")
    for e in eve_events:
        if e["label"] not in {"Recording starts", "Recording ends"}:
            print(f"  {e.get('time')}  {e['label']:30s}  onset_s={e.get('onset_s', '?')}  dur={e['duration_s']:.1f}s")
else:
    print("No EVE events available for comparison.")

In [ ]:
# ── §10d-debug  Flow signal at EVE event locations ──────────────────────────
from datetime import timedelta
import numpy as np

brp_a = session_adapted.get("brp")
if brp_a is not None:
    _flow = brp_a["Flow.40ms"]
    _fs = brp_a["fs"]
    _start = brp_a["start_dt"]
    
    abs_flow = np.abs(_flow).astype(np.float64)
    
    import pandas as _pd
    # Replicate the detector's envelope and baseline
    env_w = int(2.0 * _fs)
    bl_w = int(60.0 * _fs)
    envelope = _pd.Series(abs_flow).rolling(window=env_w, min_periods=1, center=True).mean().values
    baseline = _pd.Series(envelope).rolling(window=bl_w, min_periods=1).quantile(0.75).values
    baseline = np.maximum(baseline, 0.01)
    
    print("=== Flow envelope at each EVE event ===")
    print(f"{'EVE time':>22s}  {'onset_s':>8s}  {'envelope':>8s}  {'baseline':>8s}  {'ratio':>6s}  {'< 0.15?':>7s}  {'< 0.75?':>7s}")
    print("-" * 85)
    
    eve_events = session.get("EVE", [])
    for ev in eve_events:
        if ev["label"] in {"Recording starts", "Recording ends"}:
            continue
        # Convert EVE time to BRP sample index
        # ResMed timestamps an event at its END (see eve_event_window)
        ev_time = ev["time"]
        offset_s = (ev_time - _start).total_seconds()
        _ev_start, _ev_end = event_detection.eve_event_window(ev)
        mid_s = ((_ev_start - _start).total_seconds() + offset_s) / 2.0
        idx = int(mid_s * _fs)
        
        if 0 <= idx < len(envelope):
            env_val = envelope[idx]
            bl_val = baseline[idx]
            ratio = env_val / bl_val
            print(f"{str(ev_time):>22s}  {offset_s:>8.0f}  {env_val:>8.4f}  {bl_val:>8.4f}  {ratio:>6.3f}  {'YES' if ratio < 0.15 else 'no':>7s}  {'YES' if ratio < 0.75 else 'no':>7s}")
        else:
            print(f"{str(ev_time):>22s}  {offset_s:>8.0f}  ** out of BRP range **")
            
            

In [ ]:
# ── §10d-debug2  Detailed flow around each EVE event ──────────────────────
from datetime import timedelta
import numpy as np
import pandas as _pd

brp_a = session_adapted.get("brp")
if brp_a is not None:
    _flow = brp_a["Flow.40ms"]
    _fs = brp_a["fs"]
    _start = brp_a["start_dt"]
    
    abs_flow = np.abs(_flow).astype(np.float64)
    env_w = int(2.0 * _fs)
    bl_w = int(60.0 * _fs)
    envelope = _pd.Series(abs_flow).rolling(window=env_w, min_periods=1, center=True).mean().values
    baseline = _pd.Series(envelope).rolling(window=bl_w, min_periods=1).quantile(0.75).values
    baseline = np.maximum(baseline, 0.01)
    
    eve_events = session.get("EVE", [])
    
    for ev in eve_events:
        if ev["label"] in {"Recording starts", "Recording ends"}:
            continue
        # ResMed timestamps an event at its END; the event runs back from it.
        # Hypopneas carry no duration, so eve_event_window() assumes 10 s.
        ev_time = ev["time"]
        _ev_start, _ev_end = event_detection.eve_event_window(ev)
        dur = (_ev_end - _ev_start).total_seconds()
        offset_s = (_ev_start - _start).total_seconds()   # event start
        
        idx_ev_start = int(offset_s * _fs)
        idx_ev_end = int((offset_s + dur) * _fs)
        
        if idx_ev_start >= len(envelope):
            continue
        
        # Get min ratio during the event window
        ev_slice = envelope[idx_ev_start:min(len(envelope), idx_ev_end)]
        bl_slice = baseline[idx_ev_start:min(len(baseline), idx_ev_end)]
        ratios = ev_slice / bl_slice
        min_ratio = ratios.min() if len(ratios) > 0 else float('nan')
        
        print(f"\n{'='*70}")
        print(f"EVE: {ev_time}  {ev['label']}  dur={dur}s  (timestamp = event end)")
        print(f"  BRP offset: {offset_s:.0f}s to {offset_s+dur:.0f}s")
        print(f"  Min ratio during event: {min_ratio:.3f}")
        print(f"  {'Time offset':>12s}  {'envelope':>8s}  {'baseline':>8s}  {'ratio':>6s}  {'zone':>8s}")
        print(f"  {'-'*55}")
        
        for t_off in range(-10, int(dur) + 11, 2):
            t_abs = offset_s + t_off
            idx = int(t_abs * _fs)
            if 0 <= idx < len(envelope):
                env_val = envelope[idx]
                bl_val = baseline[idx]
                ratio = env_val / bl_val
                if t_off < 0:
                    zone = "pre"
                elif t_off > dur:
                    zone = "post"
                else:
                    zone = "EVENT"
                print(f"  {t_off:>+10d}s  {env_val:>8.4f}  {bl_val:>8.4f}  {ratio:>6.3f}  {zone:>8s}")

In [ ]:
# ── §10e  Flow waveform with custom events overlaid ──────────────────────────
from datetime import timedelta

brp_a = session_adapted.get("brp")
if brp_a is not None and not det["session_too_short"]:
    _flow   = brp_a["Flow.40ms"]
    _fs     = brp_a["fs"]
    _start  = brp_a["start_dt"]
    _step   = max(1, _flow.size // 6000)
    _fds    = _flow[::_step]
    _t      = [_start + timedelta(seconds=i*_step/_fs) for i in range(len(_fds))]

    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(_t, _fds, lw=0.4, color="steelblue", label="Flow")
    ax.axhline(0, color="gray", lw=0.5, ls="--")

    _colors = {"Apnea": "#d9534f", "Hypopnea": "#f0ad4e",
               "Obstructive Apnea (est.)": "#d9534f",
               "Central Apnea (est.)": "#5bc0de",
               "Flow Limitation": "#6a2a8c", "Snore": "#8a6a00"}
    _seen = set()
    for ev in det["all_custom_events"]:
        c = _colors.get(ev["label"], "#999")
        t0 = _start + timedelta(seconds=ev["onset_s"])
        t1 = t0 + timedelta(seconds=ev["duration_s"])
        lbl = ev["label"] if ev["label"] not in _seen else None
        ax.axvspan(t0, t1, alpha=0.25, color=c, lw=0, label=lbl)
        _seen.add(ev["label"])

    ax.set_ylabel("Flow (L/s)")
    ax.set_title(f"BRP Flow — {INSPECT_DATE} — Custom Events Overlaid")
    ax.legend(fontsize=8, loc="upper right")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

---
## 11. Indices & CTQI (single night)

Compute per-night indices, CTQI composite score, UARS assessment,
Glasgow breath analysis, and per-signal descriptive statistics.

In [ ]:
# ── §11a  Compute night indices ──────────────────────────────────────────────
from datetime import datetime

str_row = None
if df_str is not None:
    target_date = pd.Timestamp(datetime.strptime(INSPECT_DATE, "%Y%m%d").date())
    if target_date in df_str.index:
        str_row = df_str.loc[target_date].to_dict()
    else:
        # Fall back to the nearest STR date only if it is within 1 day.
        # A row from a different night would give the wrong AHI and settings.
        nearest_idx = df_str.index.get_indexer([target_date], method="nearest")[0]
        if nearest_idx >= 0:
            nearest_date = df_str.index[nearest_idx]
            if abs((nearest_date - target_date).days) <= 1:
                str_row = df_str.loc[nearest_date].to_dict()
                print(f"Note: STR row not found for {target_date.date()}, "
                      f"using adjacent date {nearest_date.date()}")
            else:
                print(f"Note: STR.edf has no row for {target_date.date()} "
                      f"(nearest is {nearest_date.date()}). STR-based values "
                      f"for this night will be missing.")

# --- Debug: str_row lookup ---
print(f"INSPECT_DATE: {INSPECT_DATE}")
print(f"target_date:  {target_date}  (type: {type(target_date)})")
print(f"df_str index dtype: {df_str.index.dtype}")
print(f"df_str index[0]:    {df_str.index[0]}  (type: {type(df_str.index[0])})")
print(f"target_date in index: {target_date in df_str.index}")
if str_row is not None:
    print(f"str_row keys: {list(str_row.keys())[:10]}...")
    print(f"str_row['S.A.MinPress']: {str_row.get('S.A.MinPress', 'MISSING')}")
else:
    print("str_row is None!")
    # Try fuzzy match to diagnose
    close = [d for d in df_str.index if abs((d - target_date).days) <= 1]
    print(f"Nearby dates in index (±1 day): {close}")
        
night_idx = indices.compute_night_indices(session_adapted, det, str_row=str_row)

ctqi = night_idx["ctqi"]
print(f"=== CTQI for {INSPECT_DATE} ===")
print(f"Score: {ctqi['ctqi']}   Grade: {ctqi['grade']}")
if not ctqi.get("insufficient"):
    print("\nComponent breakdown:")
    for k, v in ctqi.get("components", {}).items():
        wk = k.replace("_score", "")
        w  = ctqi.get("weights", {}).get(wk, "?")
        print(f"  {k:15s} = {v:6.1f}   (weight: {w})")

In [ ]:
# ── §11b  UARS assessment ────────────────────────────────────────────────────

uars = night_idx.get("uars", {})
flag = "YES" if uars.get("uars_suggestive") else "No"
print(f"=== UARS Assessment ===")
print(f"UARS Suggestive: {flag}  (score: {uars.get('uars_score', 0)}/5)")
for k, v in uars.get("uars_criteria", {}).items():
    mark = "✓" if v else "✗"
    print(f"  {mark} {k}")

In [ ]:
# ── §11c  Glasgow Index ──────────────────────────────────────────────────────

glas = night_idx.get("glasgow", {})
if glas:
    print(f"=== Glasgow Index (BRP Breath Analysis) ===")
    print(f"Breaths analyzed: {glas.get('n_breaths', 0)}")
    for k, v in glas.items():
        if k == "n_breaths":
            continue
        if isinstance(v, float) and not np.isnan(v):
            print(f"  {k:20s} = {v:.4f}")

In [ ]:
# ── §11d  Per-signal stats (compact) ─────────────────────────────────────────

sig_stats = night_idx.get("signal_stats", {})
if sig_stats:
    print("=== Per-Signal Statistics (key signals) ===")
    for sig in ["Flow.40ms", "Leak.2s", "FlowLim.2s", "SpO2.1s",
                "Pulse.1s", "MaskPress.2s", "RespRate.2s", "Snore.2s"]:
        s = sig_stats.get(sig)
        if s is None:
            continue
        print(f"\n  {sig}:")
        print(f"    mean={s.get('mean','—'):>8}  median={s.get('median','—'):>8}  "
              f"std={s.get('std','—'):>8}  "
              f"[{s.get('p5','—')} – {s.get('p95','—')}]  "
              f"n={s.get('n_valid','—')}/{s.get('n_total','—')}")

---
## 12. Batch: Event Detection + Indices (all nights)

Extends the Section 9 batch loop with custom event detection, indices,  
and CTQI for every night. **This loop is slow** (~1-2s per night for BRP analysis).  
Set `max_nights` to a small number for testing.

In [ ]:
# ── §12a  Batch loop ─────────────────────────────────────────────────────────
from tqdm.auto import tqdm
import re
import time
from datetime import datetime, timedelta

max_nights = None   # Set to e.g. 10 for testing, None for all

folders = sorted(p for p in Path(DATALOG_ROOT).iterdir()
                 if p.is_dir() and re.match(r"\d{8}", p.name))
if max_nights is not None:
    folders = folders[:max_nights]

print(f"Processing {len(folders)} nights...")
t_start = time.perf_counter()

nightly_rows_v2 = []
all_events_rows = []    # event-level data with pressure context
validation_rows_v2 = []  # per-night custom-vs-EVE comparison
uars_diagnostic_rows_v2 = []  # per-night UARS criteria and input values
batch_errors = []

for folder in tqdm(folders, desc="Batch analysis"):
    dstr = folder.name
    try:
        sess = read_session(folder)
        sess_a = adapt_session(sess)

        # STR row lookup via DatetimeIndex — must come before detect_all
        # because the UARS composite now reads RIN and OAI from str_row.
        sr = None
        if df_str is not None:
            try:
                target = pd.Timestamp(datetime.strptime(dstr, "%Y%m%d").date())
                if target in df_str.index:
                    sr = df_str.loc[target].to_dict()
            except ValueError:
                pass

        det_i = event_detection.detect_all(sess_a, str_row=sr)

        ni = indices.compute_night_indices(sess_a, det_i, str_row=sr)
        row = indices.build_nightly_summary_row(dstr, sess_a, det_i, ni, str_row=sr)
        nightly_rows_v2.append(row)

        # ── Collect event-level rows with pressure context ──
        if not det_i["session_too_short"]:
            pld = sess_a.get("pld")
            pld_press = pld.get("MaskPress.2s") if pld else None
            pld_fl = pld.get("FlowLim.2s") if pld else None
            pld_fs = pld.get("fs", 0.5) if pld else 0.5
            pld_start = pld.get("start_dt") if pld else None

            sa2 = sess_a.get("sa2")
            sa2_spo2 = sa2.get("SpO2.1s") if sa2 else None
            sa2_pulse = sa2.get("Pulse.1s") if sa2 else None
            sa2_fs = sa2.get("fs", 1.0) if sa2 else 1.0
            sa2_start = sa2.get("start_dt") if sa2 else None

            brp_start = sess_a["brp"]["start_dt"] if sess_a.get("brp") else None

            def _pld_val_at(signal_arr, abs_time, window_s=10):
                """Get median of PLD signal within ±window_s of abs_time."""
                if signal_arr is None or pld_start is None:
                    return float("nan")
                offset = (abs_time - pld_start).total_seconds()
                lo = max(0, int((offset - window_s) * pld_fs))
                hi = min(len(signal_arr), int((offset + window_s) * pld_fs))
                if lo >= hi:
                    return float("nan")
                chunk = signal_arr[lo:hi].astype(np.float64)
                valid = chunk[np.isfinite(chunk)]
                return float(np.median(valid)) if len(valid) > 0 else float("nan")

            def _sa2_val_at(signal_arr, abs_time, fs, start, window_s=10):
                """Get median of SA2 signal within ±window_s of abs_time."""
                if signal_arr is None or start is None:
                    return float("nan")
                offset = (abs_time - start).total_seconds()
                lo = max(0, int((offset - window_s) * fs))
                hi = min(len(signal_arr), int((offset + window_s) * fs))
                if lo >= hi:
                    return float("nan")
                chunk = signal_arr[lo:hi].astype(np.float64)
                valid = chunk[(chunk > 0) & np.isfinite(chunk)]
                return float(np.median(valid)) if len(valid) > 0 else float("nan")

            # Custom events (apnea, hypopnea, desat, pulse surge, snore, flow lim)
            event_lists = [
                ("events_apnea", "brp"), ("events_hypopnea", "brp"),
                ("events_desat", "sa2"), ("events_pulse_surge", "sa2"),
                ("events_snore", "pld"), ("events_flow_lim", "pld"),
            ]
            for key, ref_signal in event_lists:
                for ev in det_i.get(key, []):
                    # Determine absolute time of event
                    if "time" in ev:
                        abs_t = ev["time"]
                    elif ref_signal == "brp" and brp_start:
                        abs_t = brp_start + timedelta(seconds=ev.get("onset_s", 0))
                    elif ref_signal == "sa2" and sa2_start:
                        abs_t = sa2_start + timedelta(seconds=ev.get("onset_s", 0))
                    elif pld_start:
                        abs_t = pld_start + timedelta(seconds=ev.get("onset_s", 0))
                    else:
                        continue

                    erow = {
                        "date": dstr,
                        "source": "custom",
                        "event_type": key.replace("events_", ""),
                        "label": ev.get("label", key.replace("events_", "")),
                        "onset_s": ev.get("onset_s", 0),
                        "duration_s": ev.get("duration_s", 0),
                        "time": str(abs_t),
                        "mask_press": _pld_val_at(pld_press, abs_t),
                        "flow_lim": _pld_val_at(pld_fl, abs_t),
                        "spo2": _sa2_val_at(sa2_spo2, abs_t, sa2_fs, sa2_start),
                        "pulse": _sa2_val_at(sa2_pulse, abs_t, sa2_fs, sa2_start),
                    }
                    all_events_rows.append(erow)

            # EVE events (ResMed scored)
            eve_events = sess.get("EVE", [])
            for ev in eve_events:
                lbl = ev.get("label", "")
                if lbl in ("Recording starts", "Recording ends", ""):
                    continue
                abs_t = ev.get("time", brp_start + timedelta(seconds=ev.get("onset_s", 0)) if brp_start else None)
                if abs_t is None:
                    continue
                erow = {
                    "date": dstr,
                    "source": "eve",
                    "event_type": lbl.lower().replace(" ", "_"),
                    "label": lbl,
                    "onset_s": ev.get("onset_s", 0),
                    "duration_s": ev.get("duration_s", 0),
                    "time": str(abs_t),
                    "mask_press": _pld_val_at(pld_press, abs_t),
                    "flow_lim": _pld_val_at(pld_fl, abs_t),
                    "spo2": _sa2_val_at(sa2_spo2, abs_t, sa2_fs, sa2_start),
                    "pulse": _sa2_val_at(sa2_pulse, abs_t, sa2_fs, sa2_start),
                }
                all_events_rows.append(erow)

        # ── Per-night custom-vs-EVE validation at two tolerance windows ──
        # Uses existing event_detection.compare_with_eve (same function that
        # produced the §10d single-night F1). Safe: wrapped in outer try/except
        # so any failure here will not interrupt the main pipeline.
        if not det_i["session_too_short"]:
            resp_custom = det_i.get("events_apnea", []) + det_i.get("events_hypopnea", [])
            eve_raw = sess.get("EVE", [])
            # Filter EVE boilerplate markers before comparison
            eve_for_cmp = [e for e in eve_raw
                           if e.get("label", "") not in
                              ("Recording starts", "Recording ends", "")]

            for tol in (15.0, 30.0):
                cmp_result = event_detection.compare_with_eve(
                    resp_custom, eve_for_cmp, tolerance_s=tol
                )
                validation_rows_v2.append({
                    "date":         dstr,
                    "tolerance_s":  tol,
                    "custom_total": cmp_result["custom_total"],
                    "eve_total":    cmp_result["eve_total"],
                    "matched":      cmp_result["matched"],
                    "custom_only":  cmp_result["custom_only"],
                    "eve_only":     cmp_result["eve_only"],
                    "precision":    cmp_result["precision"],
                    "recall":       cmp_result["recall"],
                    "f1":           cmp_result["f1"],
                })

        # ── Per-night UARS criterion diagnostic ──────────────────────────
        # Captures the five raw inputs and the five criterion booleans so
        # that threshold calibration can be evaluated post-hoc without
        # modifying assess_uars_composite. Safe: independent try/except
        # so any failure here won't interrupt the batch.
        try:
            # Raw inputs — same sources used by detect_all.
            # RIN and OAI come from the STR row (not session metadata);
            # fl_frac_elevated replaces fl_mean to match the Option A fix
            # in assess_uars_composite.
            rin_val = float(sr.get("RIN", float("nan"))) if sr is not None else float("nan")
            oai_val = float(sr.get("OAI", float("nan"))) if sr is not None else float("nan")
            fl_val = det_i.get("fl_metrics", {}).get("fl_frac_elevated", float("nan"))
            pci_val = det_i.get("pulse_metrics", {}).get("PCI_6bpm", float("nan"))

            # Recompute pressure slope in the loop (no change to detect_all)
            pslope_val = float("nan")
            pld_sig = sess_a.get("pld")
            if pld_sig is not None:
                mp = pld_sig.get("MaskPress.2s")
                if mp is not None and mp.size > 0:
                    pslope_val = event_detection.compute_pressure_slope(
                        mp, pld_sig.get("fs", 0.5)
                    )

            # Pull the per-criterion booleans directly from the stored assessment
            uars_asmt = det_i.get("uars_assessment", {})
            crit = uars_asmt.get("uars_criteria", {})

            uars_diagnostic_rows_v2.append({
                "date":              dstr,
                "rin":               rin_val,
                "fl_frac_elevated":  fl_val,
                "oai":               oai_val,
                "pressure_slope":    pslope_val,
                "pci":               pci_val,
                "crit_rin_elevated":      bool(crit.get("rin_elevated", False)),
                "crit_fl_elevated":       bool(crit.get("fl_elevated", False)),
                "crit_oai_low":           bool(crit.get("oai_low", False)),
                "crit_pressure_rising":   bool(crit.get("pressure_rising", False)),
                "crit_pci_elevated":      bool(crit.get("pci_elevated", False)),
                "uars_score":        uars_asmt.get("uars_score", 0),
                "uars_suggestive":   uars_asmt.get("uars_suggestive", False),
            })
        except Exception as _e:
            pass  # UARS diagnostic is optional; skip this night silently

    except Exception as e:
        batch_errors.append((dstr, str(e)))

df_nightly_v2 = pd.DataFrame(nightly_rows_v2)
df_all_events = pd.DataFrame(all_events_rows) if all_events_rows else pd.DataFrame()
df_validation_v2 = pd.DataFrame(validation_rows_v2) if validation_rows_v2 else pd.DataFrame()
df_uars_diag_v2 = pd.DataFrame(uars_diagnostic_rows_v2) if uars_diagnostic_rows_v2 else pd.DataFrame()

elapsed = time.perf_counter() - t_start
print(f"\nDone. {len(df_nightly_v2)} nights, {len(batch_errors)} errors.")
print(f"Elapsed: {elapsed:.1f}s  ({elapsed/max(1, len(df_nightly_v2)):.1f}s per night)")
print(f"Event-level rows: {len(df_all_events)}")
print(f"Validation rows:  {len(df_validation_v2)}")
print(f"UARS diagnostic rows: {len(df_uars_diag_v2)}")
if batch_errors:
    print("First errors:")
    for d, e in batch_errors[:5]:
        print(f"  {d}: {e}")

In [ ]:
# ── §12b  Save enhanced nightly summary ──────────────────────────────────────

# ── §12b  Save enhanced nightly summary ──────────────────────────────────────

csv_v2 = Path(".") / "nightly_summary_v2.csv"
df_nightly_v2.to_csv(csv_v2, index=False)
print(f"Saved {len(df_nightly_v2)} rows to {csv_v2}")

# Validation table
if len(df_validation_v2) > 0:
    val_csv = Path(".") / "validation_v2.csv"
    df_validation_v2.to_csv(val_csv, index=False)
    print(f"Saved {len(df_validation_v2)} validation rows to {val_csv}")

# UARS diagnostic table
if len(df_uars_diag_v2) > 0:
    uars_csv = Path(".") / "uars_diagnostic_v2.csv"
    df_uars_diag_v2.to_csv(uars_csv, index=False)
    print(f"Saved {len(df_uars_diag_v2)} UARS diagnostic rows to {uars_csv}")

show_cols = ["date", "duration_min", "str_AHI", "custom_ahi", "CTQI", "CTQI_grade",
             "ODI_3", "PCI_6bpm", "fl_mean", "snore_index",
             "uars_suggestive", "glasgow_overall"]
avail_cols = [c for c in show_cols if c in df_nightly_v2.columns]
df_nightly_v2[avail_cols].head(15)

In [ ]:
# ── §12c  Plot CTQI over time ────────────────────────────────────────────────

df_ctqi = df_nightly_v2.dropna(subset=["CTQI"]).copy()
df_ctqi["date_dt"] = pd.to_datetime(df_ctqi["date"])

if len(df_ctqi) > 0:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.scatter(df_ctqi["date_dt"], df_ctqi["CTQI"], s=12, alpha=0.5, color="#1a6a2a")
    if len(df_ctqi) >= 7:
        roll = df_ctqi.set_index("date_dt")["CTQI"].rolling(7, min_periods=3).median()
        ax.plot(roll.index, roll.values, lw=2, color="#1a6a2a", label="7-day median")
    ax.axhline(90, color="#1a8a1a", ls=":", alpha=0.5, label="Excellent")
    ax.axhline(75, color="#b08a00", ls=":", alpha=0.5, label="Good")
    ax.set_ylabel("CTQI (0–100)")
    ax.set_title("CPAP Therapy Quality Index Over Time")
    ax.legend(fontsize=8)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    print("\nCTQI Grade Distribution:")
    print(df_nightly_v2["CTQI_grade"].value_counts().to_string())
else:
    print("No CTQI values to plot.")

---
## 13. Settings Changes

Detect therapy settings changes from STR signals.  
Compare key metrics before vs. after each change using Mann-Whitney U.  
P-values are Holm-corrected across the metrics tested for each change, and a
direction (improved / worsened) is only reported when the corrected p < 0.05.

In [ ]:
# ── §13a  Detect changes & run full analysis ─────────────────────────────────

ctqi_series = None
if "CTQI" in df_nightly_v2.columns:
    ctqi_series = df_nightly_v2.set_index("date")["CTQI"]

change_summary = settings_changes.settings_change_summary(
    df_str, ctqi_series=ctqi_series
)

n_ch = change_summary['n_changes']
n_unconf = sum(1 for ba in change_summary.get("before_after", [])
               if not ba.get("confounded") and not ba.get("insufficient"))
n_insuff = sum(1 for ba in change_summary.get("before_after", [])
               if ba.get("insufficient"))

print(f"=== Settings Changes ===")
print(f"Change dates detected: {n_ch}")
print(f"  Unconfounded with sufficient data: {n_unconf}")
print(f"  Insufficient data (skipped):       {n_insuff}")
print(f"  Confounded windows:                {n_ch - n_unconf - n_insuff}")
if n_ch > 0:
    display(change_summary["changes_df"])

In [ ]:
# ── §13b  Before / after comparisons ─────────────────────────────────────────
# Show full detail for unconfounded changes.
# For confounded changes, show only metrics with |effect| > 0.3.

EFFECT_THRESHOLD = 0.3   # practical significance cutoff

for ba in change_summary.get("before_after", []):
    cd = ba["change_date"]
    settings_str = ", ".join(
        f"{sc['signal']}: {sc['old_value']} → {sc['new_value']}"
        for sc in ba.get("settings_changed", []))

    if ba.get("insufficient"):
        print(f"\n--- {cd} --- {settings_str}")
        print("  ⚠ Insufficient data for comparison (skipped)")
        continue

    is_conf = ba.get("confounded", False)

    # Filter comparisons to significant effects if confounded
    comps = ba.get("comparisons", {})
    if is_conf:
        sig_comps = {m: c for m, c in comps.items()
                     if not c.get("insufficient")
                     and abs(c.get("effect_size", 0)) >= EFFECT_THRESHOLD}
        if not sig_comps:
            print(f"\n--- {cd} [confounded] --- {settings_str}")
            print(f"  Before: {ba['before_n']} / After: {ba['after_n']} nights — "
                  f"no metrics with |effect| >= {EFFECT_THRESHOLD}")
            continue
        show_comps = sig_comps
    else:
        show_comps = {m: c for m, c in comps.items()
                      if not c.get("insufficient")}

    tag = " [confounded]" if is_conf else ""
    print(f"\n--- {cd}{tag} --- {settings_str}")
    print(f"  Before: {ba['before_n']} nights, After: {ba['after_n']} nights")
    print(f"  {'Metric':20s} {'Before':>8s} {'After':>8s} {'p-val':>8s} {'p-Holm':>8s} {'Effect':>8s} {'Dir':>16s}")
    for metric, c in show_comps.items():
        print(f"  {metric:20s} {c.get('before_median',0):8.3f} {c.get('after_median',0):8.3f} "
              f"{c.get('p_value',0):8.4f} {c.get('p_adjusted',float('nan')):8.4f} "
              f"{c.get('effect_size',0):8.3f} {c.get('direction','—'):>16s}")

In [ ]:
# ── §13c  AHI + CTQI with change markers ────────────────────────────────────

change_dates = [pd.Timestamp(cd).to_pydatetime()
                for cd in change_summary.get("change_dates", [])]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# AHI
ax = axes[0]
ax.scatter(df_active.index, df_active["AHI"], s=8, alpha=0.5, color="#d9534f")
if len(df_active) >= 7:
    _r = df_active["AHI"].rolling(7, min_periods=3).median()
    ax.plot(df_active.index, _r, lw=2, color="#d9534f")
for cd in change_dates:
    ax.axvline(cd, color="#333", ls=":", alpha=0.6)
ax.set_ylabel("AHI (events/hr)")
ax.set_title("AHI Over Time with Settings Changes")

# CTQI
ax = axes[1]
if len(df_ctqi) > 0:
    ax.scatter(df_ctqi["date_dt"], df_ctqi["CTQI"], s=8, alpha=0.5, color="#1a6a2a")
    if len(df_ctqi) >= 7:
        _r = df_ctqi.set_index("date_dt")["CTQI"].rolling(7, min_periods=3).median()
        ax.plot(_r.index, _r.values, lw=2, color="#1a6a2a")
    for cd in change_dates:
        ax.axvline(cd, color="#333", ls=":", alpha=0.6)
ax.set_ylabel("CTQI (0–100)")
ax.set_title("CTQI Over Time with Settings Changes")

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

---
## 14. Pulse-Flow Correlation

Cross-night Spearman correlations between PCI, flow limitation, AHI, and snore.  
Aligned 4-panel signal plot (flow + EVE events, SpO2, pulse, FlowLim) for the inspected night.  
Per-event co-occurrence analysis: what fraction of pulse surges temporally coincide  
with EVE respiratory events (±30s) and elevated FlowLim (>0.3)?

In [ ]:
# ── §14a  Cross-night correlation matrix ─────────────────────────────────────

corr_cols_v2 = ["custom_ahi", "PCI_6bpm", "fl_mean", "snore_index", "ODI_3"]
avail = [c for c in corr_cols_v2 if c in df_nightly_v2.columns]

if len(avail) >= 2:
    corr_df = df_nightly_v2[avail].corr(method="spearman").round(3)
    print("Cross-night metric correlations (Spearman rank):")
    print(corr_df.to_string())
    print(f"\n(n = {len(df_nightly_v2.dropna(subset=avail, how='all'))} nights with at least one metric)")

    # Heatmap
    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(corr_df.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(avail)))
    ax.set_yticks(range(len(avail)))
    short_labels = [c.replace("custom_", "").replace("_6bpm", "")
                    .replace("_index", "") for c in avail]
    ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(short_labels, fontsize=9)
    for i in range(len(avail)):
        for j in range(len(avail)):
            val = corr_df.values[i, j]
            color = "white" if abs(val) > 0.6 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=9, color=color)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Spearman r")
    ax.set_title("Cross-Night Metric Correlations (Spearman)")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough metrics for correlation.")

In [ ]:
# ── §14b  Aligned flow / SpO2 / pulse / FlowLim plot ────────────────────────

sa2_a = session_adapted.get("sa2")
brp_a = session_adapted.get("brp")
pld_a = session_adapted.get("pld")

if sa2_a is not None and brp_a is not None and not det["session_too_short"]:
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

    # Panel 1: Flow + EVE event markers
    _fl = brp_a["Flow.40ms"]
    _fs = brp_a["fs"]
    _st = brp_a["start_dt"]
    _step = max(1, _fl.size // 5000)
    _fds = _fl[::_step]
    _t = [_st + timedelta(seconds=i*_step/_fs) for i in range(len(_fds))]
    axes[0].plot(_t, _fds, lw=0.3, color="steelblue")
    # Overlay EVE events on flow panel
    eve_events = session.get("EVE", [])
    eve_colors = {"Obstructive Apnea": "#d9534f", "Hypopnea": "#f0ad4e",
                  "Central Apnea": "#5bc0de", "RERA": "#9b59b6",
                  "Unclassified Apnea": "#aaaaaa"}
    eve_labels_shown = set()
    for ev in eve_events:
        lbl = ev.get("label", "")
        if lbl in ("Recording starts", "Recording ends"):
            continue
        # ResMed timestamps an event at its END; shade the span before it
        if "time" in ev:
            t0, t1 = event_detection.eve_event_window(ev)
        else:
            t1 = _st + timedelta(seconds=ev.get("onset_s", 0))
            t0 = t1 - timedelta(seconds=ev.get("duration_s") or 10.0)
        clr = eve_colors.get(lbl, "#888888")
        show_lbl = lbl if lbl not in eve_labels_shown else None
        axes[0].axvspan(t0, t1, alpha=0.35, color=clr, label=show_lbl)
        eve_labels_shown.add(lbl)
    axes[0].set_ylabel("Flow (L/s)")
    axes[0].set_title(f"Aligned Signals — {INSPECT_DATE}")
    if eve_labels_shown:
        axes[0].legend(fontsize=7, loc="upper right", ncol=3)

    # Panel 2: SpO2 + desat events
    _spo2 = sa2_a["SpO2.1s"].astype(np.float64).copy()
    _spo2[_spo2 <= 0] = np.nan
    _fss = sa2_a["fs"]
    _sts = sa2_a["start_dt"]
    _ts = [_sts + timedelta(seconds=i/_fss) for i in range(len(_spo2))]
    axes[1].plot(_ts, _spo2, lw=0.5, color="#1a6a2a")
    axes[1].set_ylabel("SpO2 (%)")
    axes[1].set_ylim(80, 100)
    for ev in det.get("events_desat", []):
        t0 = _sts + timedelta(seconds=ev["onset_s"])
        t1 = t0 + timedelta(seconds=ev["duration_s"])
        axes[1].axvspan(t0, t1, alpha=0.3, color="#d9534f")

    # Panel 3: Pulse + pulse surge events
    _pulse = sa2_a["Pulse.1s"].astype(np.float64).copy()
    _pulse[_pulse <= 0] = np.nan
    _tp = [_sts + timedelta(seconds=i/_fss) for i in range(len(_pulse))]
    axes[2].plot(_tp, _pulse, lw=0.5, color="#c04000")
    axes[2].set_ylabel("Pulse (bpm)")
    for ev in det.get("events_pulse_surge", []):
        t0 = _sts + timedelta(seconds=ev["onset_s"])
        t1 = t0 + timedelta(seconds=ev["duration_s"])
        axes[2].axvspan(t0, t1, alpha=0.3, color="#f0ad4e")

    # Panel 4: FlowLim.2s from PLD
    if pld_a is not None and pld_a.get("FlowLim.2s") is not None:
        _flim = pld_a["FlowLim.2s"].astype(np.float64)
        _fps = pld_a["fs"]
        _stp = pld_a["start_dt"]
        _tfl = [_stp + timedelta(seconds=i/_fps) for i in range(len(_flim))]
        axes[3].plot(_tfl, _flim, lw=0.5, color="#8e44ad")
        axes[3].axhline(0.3, color="#8e44ad", ls=":", alpha=0.5, label="Threshold (0.3)")
        axes[3].set_ylabel("FlowLim")
        axes[3].set_ylim(-0.05, 1.05)
        axes[3].legend(fontsize=7, loc="upper right")
    else:
        axes[3].text(0.5, 0.5, "FlowLim.2s not available", transform=axes[3].transAxes,
                     ha="center", va="center", fontsize=10, color="gray")
        axes[3].set_ylabel("FlowLim")

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("SA2 data not available for aligned signal plot.")

In [ ]:
# ── §14c  Pulse surge co-occurrence analysis ─────────────────────────────────
# For each night in the batch, check temporal co-occurrence between:
#   - Pulse surges ↔ EVE respiratory events (±30s)
#   - Pulse surges ↔ FlowLim > 0 (±10s)
#   - Pulse surges ↔ SpO2 desaturations (±30s)
#   - EVE events ↔ FlowLim > 0 (±10s)
#   - EVE events ↔ SpO2 desaturations (±30s)

import time as _time

CO_OCC_WINDOW = 30.0   # seconds, ±window for event matching

cooc_rows = []
max_nights_cooc = None   # Set to None for all nights

_cooc_folders = sorted(p for p in Path(DATALOG_ROOT).iterdir()
                       if p.is_dir() and re.match(r"\d{8}", p.name))
if max_nights_cooc is not None:
    _cooc_folders = _cooc_folders[:max_nights_cooc]

print(f"Scanning {len(_cooc_folders)} nights for co-occurrence...")
_t0 = _time.perf_counter()

for folder in tqdm(_cooc_folders, desc="Co-occurrence"):
    dstr = folder.name
    try:
        sess = read_session(folder)
        sess_a = adapt_session(sess)

        # Need SA2 for pulse surges and desats
        if sess_a.get("sa2") is None:
            continue

        det_i = event_detection.detect_all(sess_a)
        if det_i["session_too_short"]:
            continue

        ps_events = det_i.get("events_pulse_surge", [])
        ds_events = det_i.get("events_desat", [])

        # EVE respiratory events (exclude Recording starts/ends)
        eve_raw = sess.get("EVE", [])
        eve_resp = [e for e in eve_raw
                    if e.get("label", "") not in ("Recording starts", "Recording ends", "")]

        # FlowLim from PLD
        pld = sess_a.get("pld")
        fl_arr = None
        fl_fs = 0.5
        fl_start = None
        if pld is not None and pld.get("FlowLim.2s") is not None:
            fl_arr = pld["FlowLim.2s"].astype(np.float64)
            fl_fs = pld.get("fs", 0.5)
            fl_start = pld.get("start_dt")

        sa2_start = sess_a["sa2"]["start_dt"]

        # Helper: check if FlowLim > 0 within ±10s of a given absolute time
        def _has_fl(abs_time):
            if fl_arr is None or fl_start is None:
                return False
            offset_s = (abs_time - fl_start).total_seconds()
            lo = max(0, int((offset_s - 30) * fl_fs))
            hi = min(len(fl_arr), int((offset_s + 30) * fl_fs))
            return lo < hi and np.nanmax(fl_arr[lo:hi]) > 0.0

        # Helper: absolute midpoint of an event. Custom events carry 'time'
        # (their start). EVE events are timestamped at their END.
        def _abs_mid(ev, is_eve=False):
            if is_eve:
                a, b = event_detection.eve_event_window(ev)
                return a + (b - a) / 2
            return ev["time"] + timedelta(seconds=ev.get("duration_s", 0) / 2.0)

        # Helper: check if two event lists have a match within ±window,
        # comparing absolute clock times (files start at different times)
        def _has_match(ev_a, events_b, window, a_is_eve=False, b_is_eve=False):
            mid_a = _abs_mid(ev_a, a_is_eve)
            for eb in events_b:
                if abs((mid_a - _abs_mid(eb, b_is_eve)).total_seconds()) <= window:
                    return True
            return False

        n_ps = len(ps_events)
        n_eve = len(eve_resp)
        n_ds = len(ds_events)

        # Pulse surge → EVE, FlowLim, Desat
        ps_w_eve, ps_w_fl, ps_w_ds = 0, 0, 0
        for ps in ps_events:
            ps_onset = ps["onset_s"]
            ps_dur = ps.get("duration_s", 0)
            ps_abs = sa2_start + timedelta(seconds=ps_onset)
            if _has_match(ps, eve_resp, CO_OCC_WINDOW, b_is_eve=True):
                ps_w_eve += 1
            if _has_fl(ps_abs):
                ps_w_fl += 1
            if _has_match(ps, ds_events, CO_OCC_WINDOW):
                ps_w_ds += 1

        # EVE → Pulse surge, FlowLim, Desat
        eve_w_ps, eve_w_fl, eve_w_ds = 0, 0, 0
        for ev in eve_resp:
            ev_onset = ev.get("onset_s", 0)
            ev_dur = ev.get("duration_s", 0)
            ev_abs = _abs_mid(ev, is_eve=True)
            if _has_match(ev, ps_events, CO_OCC_WINDOW, a_is_eve=True):
                eve_w_ps += 1
            if _has_fl(ev_abs):
                eve_w_fl += 1
            if _has_match(ev, ds_events, CO_OCC_WINDOW, a_is_eve=True):
                eve_w_ds += 1

        # Desat → Pulse surge, EVE, FlowLim
        ds_w_ps, ds_w_eve, ds_w_fl = 0, 0, 0
        for d in ds_events:
            d_onset = d["onset_s"]
            d_dur = d.get("duration_s", 0)
            d_abs = sa2_start + timedelta(seconds=d_onset)
            if _has_match(d, ps_events, CO_OCC_WINDOW):
                ds_w_ps += 1
            if _has_match(d, eve_resp, CO_OCC_WINDOW, b_is_eve=True):
                ds_w_eve += 1
            if _has_fl(d_abs):
                ds_w_fl += 1

        def _pct(n, total):
            return round(100 * n / total, 1) if total > 0 else 0.0

        cooc_rows.append({
            "date": dstr,
            "n_ps": n_ps, "n_eve": n_eve, "n_ds": n_ds,
            # Pulse surge co-occurrence
            "ps_w_eve": ps_w_eve, "ps_w_fl": ps_w_fl, "ps_w_ds": ps_w_ds,
            "ps%eve": _pct(ps_w_eve, n_ps), "ps%fl": _pct(ps_w_fl, n_ps), "ps%ds": _pct(ps_w_ds, n_ps),
            # EVE co-occurrence
            "eve_w_ps": eve_w_ps, "eve_w_fl": eve_w_fl, "eve_w_ds": eve_w_ds,
            "eve%ps": _pct(eve_w_ps, n_eve), "eve%fl": _pct(eve_w_fl, n_eve), "eve%ds": _pct(eve_w_ds, n_eve),
            # Desat co-occurrence
            "ds_w_ps": ds_w_ps, "ds_w_eve": ds_w_eve, "ds_w_fl": ds_w_fl,
            "ds%ps": _pct(ds_w_ps, n_ds), "ds%eve": _pct(ds_w_eve, n_ds), "ds%fl": _pct(ds_w_fl, n_ds),
        })
    except Exception:
        continue

_elapsed = _time.perf_counter() - _t0

if cooc_rows:
    df_cooc_raw = pd.DataFrame(cooc_rows)
    n_sa2_files = len(df_cooc_raw)
    # Filter to nights where oximeter was actually worn (pulse data exists)
    df_cooc = df_cooc_raw[(df_cooc_raw["n_ps"] > 0) | (df_cooc_raw["n_ds"] > 0)].copy()
    n_nights = len(df_cooc)

    # Aggregate totals
    t_ps = df_cooc["n_ps"].sum()
    t_eve = df_cooc["n_eve"].sum()
    t_ds = df_cooc["n_ds"].sum()

    n_with_eve = (df_cooc["n_eve"] > 0).sum()
    n_with_ds = (df_cooc["n_ds"] > 0).sum()

    print(f"\nElapsed: {_elapsed:.1f}s  ({_elapsed/n_sa2_files:.1f}s per night)")

    print(f"\nData availability:")
    print(f"  Nights with SA2 file:       {n_sa2_files}")
    print(f"  Nights with oximeter worn:  {n_nights}")
    print(f"  Nights with EVE events:     {n_with_eve}")
    print(f"  Nights with desaturations:  {n_with_ds}")

    print(f"\n=== Pulse Surge Co-Occurrence ({n_nights} nights with oximeter) ===")
    print(f"  Total pulse surges: {t_ps}")
    if t_ps > 0:
        print(f"    with EVE event (±30s): {df_cooc['ps_w_eve'].sum():4d}  ({100*df_cooc['ps_w_eve'].sum()/t_ps:.1f}%)")
        print(f"    with FlowLim > 0:      {df_cooc['ps_w_fl'].sum():4d}  ({100*df_cooc['ps_w_fl'].sum()/t_ps:.1f}%)")
        print(f"    with SpO2 desat:       {df_cooc['ps_w_ds'].sum():4d}  ({100*df_cooc['ps_w_ds'].sum()/t_ps:.1f}%)")

    print(f"\n=== EVE Event Co-Occurrence ({n_with_eve} nights with EVE events) ===")
    print(f"  Total EVE respiratory events: {t_eve}")
    if t_eve > 0:
        print(f"    with pulse surge (±30s):  {df_cooc['eve_w_ps'].sum():4d}  ({100*df_cooc['eve_w_ps'].sum()/t_eve:.1f}%)")
        print(f"    with FlowLim > 0:         {df_cooc['eve_w_fl'].sum():4d}  ({100*df_cooc['eve_w_fl'].sum()/t_eve:.1f}%)")
        print(f"    with SpO2 desat (±30s):   {df_cooc['eve_w_ds'].sum():4d}  ({100*df_cooc['eve_w_ds'].sum()/t_eve:.1f}%)")

    print(f"\n=== SpO2 Desaturation Co-Occurrence ({n_with_ds} nights with desats) ===")
    print(f"  Total desaturation events: {t_ds}")
    if t_ds > 0:
        print(f"    with pulse surge (±30s):  {df_cooc['ds_w_ps'].sum():4d}  ({100*df_cooc['ds_w_ps'].sum()/t_ds:.1f}%)")
        print(f"    with EVE event (±30s):    {df_cooc['ds_w_eve'].sum():4d}  ({100*df_cooc['ds_w_eve'].sum()/t_ds:.1f}%)")
        print(f"    with FlowLim > 0:         {df_cooc['ds_w_fl'].sum():4d}  ({100*df_cooc['ds_w_fl'].sum()/t_ds:.1f}%)")

    print(f"\n  (FlowLim co-occurrence uses any FlowLim > 0 within ±30s)")
    print(f"  (EVE/pulse/desat co-occurrence uses ±{CO_OCC_WINDOW:.0f}s window on event midpoints)")

    # Per-night detail table (already filtered to oximeter-worn nights)
    print(f"\n--- Per-Night Detail ({n_nights} nights with oximeter) ---")
    detail_cols = ["date", "n_ps", "n_eve", "n_ds",
                   "ps%eve", "ps%fl", "ps%ds",
                   "eve%ps", "eve%fl", "eve%ds",
                   "ds%ps", "ds%eve", "ds%fl"]
    print(df_cooc[detail_cols].to_string(index=False))
else:
    print(f"\nElapsed: {_elapsed:.1f}s")
    print("No nights with SA2 data available for co-occurrence analysis.")

---
## 15. Sleep Schedule Analysis

Bedtime, wake time, duration patterns. Weeknight vs. weekend comparisons.
Missed nights. AHI/CTQI trends by day of week and bedtime.

In [ ]:
# ── §15a  Sleep schedule overview ────────────────────────────────────────────

# Build schedule data from STR (no re-reading needed)
df_sched = df_active.copy()

# MaskOn_first = minutes from midnight when mask first put on
# MaskOff_last = minutes from midnight when mask last removed
# Duration = therapy minutes

if "MaskOn_first" in df_sched.columns and "MaskOff_last" in df_sched.columns:
    # Convert minutes-from-midnight to clock times
    def _min_to_time_str(m):
        """Convert minutes-from-noon to HH:MM clock time string."""
        if pd.isna(m) or m < 0:
            return ""
        # ResMed day boundary is noon. MaskOn/MaskOff are minutes from 12:00.
        total_min = m + 720  # shift to minutes from midnight
        h = int(total_min // 60) % 24
        mi = int(total_min % 60)
        return f"{h:02d}:{mi:02d}"

    df_sched["bedtime_str"] = df_sched["MaskOn_first"].apply(_min_to_time_str)
    df_sched["waketime_str"] = df_sched["MaskOff_last"].apply(_min_to_time_str)

    # Convert to real clock hours (minutes from noon → hours from midnight)
    bed_h = (df_sched["MaskOn_first"] + 720) / 60.0  # clock hour (may exceed 24)
    bed_h_wrapped = bed_h % 24.0
    # Bedtime normalized: hours after 6 PM. So 10 PM = 4.0, 1 AM = 7.0
    df_sched["bedtime_norm"] = bed_h_wrapped.apply(lambda h: h - 18.0 if h >= 18.0 else h + 6.0)
    df_sched["bedtime_clock"] = bed_h_wrapped

    # Wake hour (clock)
    wake_h = (df_sched["MaskOff_last"] + 720) / 60.0
    df_sched["wake_hour"] = wake_h % 24.0

    # Day of week
    df_sched["weekday"] = df_sched.index.day_name()
    df_sched["dow_num"] = df_sched.index.dayofweek  # Mon=0, Sun=6
    df_sched["is_weekend"] = df_sched["dow_num"] >= 4  # Fri/Sat night

    print("=== Sleep Schedule Summary ===")
    print(f"Nights with therapy: {len(df_sched)}")

    # Bedtime stats
    bed_valid = df_sched["MaskOn_first"].dropna()
    if len(bed_valid) > 0:
        med_bed = bed_valid.median()
        print(f"\nBedtime (mask on):")
        print(f"  Median: {_min_to_time_str(med_bed)}  "
              f"(range: {_min_to_time_str(bed_valid.min())} – {_min_to_time_str(bed_valid.max())})")
        print(f"  Std dev: {bed_valid.std()/60:.1f} hours")

    # Wake time stats
    wake_valid = df_sched["MaskOff_last"].dropna()
    if len(wake_valid) > 0:
        med_wake = wake_valid.median()
        print(f"\nWake time (mask off):")
        print(f"  Median: {_min_to_time_str(med_wake)}  "
              f"(range: {_min_to_time_str(wake_valid.min())} – {_min_to_time_str(wake_valid.max())})")

    # Duration stats
    dur_valid = df_sched["Duration"].dropna()
    if len(dur_valid) > 0:
        print(f"\nTherapy duration:")
        print(f"  Median: {dur_valid.median()/60:.1f} hrs  "
              f"(range: {dur_valid.min()/60:.1f} – {dur_valid.max()/60:.1f} hrs)")
        print(f"  Nights < 6 hrs: {(dur_valid < 360).sum()}  "
              f"({100*(dur_valid < 360).sum()/len(dur_valid):.0f}%)")
        print(f"  Nights < 4 hrs: {(dur_valid < 240).sum()}  "
              f"({100*(dur_valid < 240).sum()/len(dur_valid):.0f}%)")
else:
    print("MaskOn/MaskOff not available in STR data.")

In [ ]:
# ── §15b  Missed nights ─────────────────────────────────────────────────────

# Identify gaps in the date range
all_dates = pd.date_range(df_str.index.min(), df_str.index.max(), freq="D")
therapy_dates = set(df_active.index.normalize())
missed_dates = [d for d in all_dates if d not in therapy_dates]

print(f"=== Missed Nights ===")
print(f"Date range: {all_dates[0].date()} to {all_dates[-1].date()} ({len(all_dates)} days)")
print(f"Nights with therapy (>30 min): {len(df_active)}")
print(f"Missed nights: {len(missed_dates)} ({100*len(missed_dates)/len(all_dates):.1f}%)")

if missed_dates:
    # Find streaks of missed nights
    missed_series = pd.Series(missed_dates)
    gaps = missed_series.diff().dt.days
    streak_starts = [0] + list(gaps[gaps > 1].index)
    streaks = []
    for i, start_idx in enumerate(streak_starts):
        end_idx = streak_starts[i+1] - 1 if i + 1 < len(streak_starts) else len(missed_series) - 1
        streaks.append((missed_series.iloc[start_idx].date(),
                        missed_series.iloc[end_idx].date(),
                        end_idx - start_idx + 1))
    streaks.sort(key=lambda x: -x[2])
    print(f"\nLongest missed streaks:")
    for s, e, n in streaks[:10]:
        print(f"  {s} to {e}  ({n} night{'s' if n > 1 else ''})")

    # Missed by day of week
    missed_dow = pd.Series(missed_dates).dt.day_name().value_counts()
    print(f"\nMissed nights by day of week:")
    for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
        print(f"  {day:10s}: {missed_dow.get(day, 0)}")

In [ ]:
# ── §15c  Weeknight vs. weekend comparison ──────────────────────────────────

if "is_weekend" in df_sched.columns:
    wknight = df_sched[~df_sched["is_weekend"]]
    wkend = df_sched[df_sched["is_weekend"]]

    print(f"=== Weeknight vs. Weekend ===")
    print(f"Weeknights (Sun-Thu): {len(wknight)}   Weekends (Fri-Sat): {len(wkend)}")

    compare_cols = ["Duration", "AHI", "Leak.50", "MaskPress.50"]
    if "MaskOn_first" in df_sched.columns:
        compare_cols = ["MaskOn_first", "MaskOff_last", "Duration"] + compare_cols[1:]

    print(f"\n  {'Metric':18s} {'Weeknight':>12s} {'Weekend':>12s} {'Diff':>10s}")
    for col in compare_cols:
        if col not in df_sched.columns:
            continue
        wn_med = wknight[col].median()
        we_med = wkend[col].median()
        diff = we_med - wn_med

        # Format nicely
        if "MaskOn" in col or "MaskOff" in col:
            label = "Bedtime" if "On" in col else "Wake time"
            print(f"  {label:18s} {_min_to_time_str(wn_med):>12s} {_min_to_time_str(we_med):>12s} "
                  f"{diff/60:+.1f} hrs")
        elif col == "Duration":
            print(f"  {'Duration (hrs)':18s} {wn_med/60:12.1f} {we_med/60:12.1f} {diff/60:+10.1f}")
        else:
            print(f"  {col:18s} {wn_med:12.2f} {we_med:12.2f} {diff:+10.2f}")

In [ ]:
# ── §15d  AHI by day of week + bedtime correlation ──────────────────────────

if "dow_num" in df_sched.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # AHI by day of week
    ax = axes[0]
    dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    dow_medians = df_sched.groupby("weekday")["AHI"].median().reindex(dow_order)
    ax.bar(range(7), dow_medians.values, color="#d9534f", alpha=0.7)
    ax.set_xticks(range(7))
    ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], fontsize=8)
    ax.set_ylabel("Median AHI")
    ax.set_title("AHI by Day of Week")

    # Duration by day of week
    ax = axes[1]
    dow_dur = df_sched.groupby("weekday")["Duration"].median().reindex(dow_order) / 60.0
    ax.bar(range(7), dow_dur.values, color="#5bc0de", alpha=0.7)
    ax.set_xticks(range(7))
    ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], fontsize=8)
    ax.set_ylabel("Median Duration (hrs)")
    ax.set_title("Therapy Duration by Day of Week")

    # AHI vs bedtime scatter
    ax = axes[2]
    bed_ahi = df_sched.dropna(subset=["bedtime_norm", "AHI"])
    if len(bed_ahi) > 0:
        ax.scatter(bed_ahi["bedtime_norm"], bed_ahi["AHI"], s=8, alpha=0.3, color="#333")
        ax.set_xlabel("Bedtime (hours after 6 PM)")
        ax.set_ylabel("AHI")
        ax.set_title("AHI vs. Bedtime")
        # Add correlation
        from scipy.stats import spearmanr
        r, p = spearmanr(bed_ahi["bedtime_norm"], bed_ahi["AHI"])
        ax.text(0.05, 0.95, f"Spearman r={r:.3f}\np={p:.3f}",
                transform=ax.transAxes, va="top", fontsize=8,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    plt.tight_layout()
    plt.show()

---
## 16. Pressure-Event Analysis

What pressure was the machine delivering when each event occurred?
Pressure-stratified event rates and statistics.

In [ ]:
# ── §16a  Pressure distribution at events ────────────────────────────────────

if len(df_all_events) > 0:
    df_ev = df_all_events.copy()
    df_ev["mask_press"] = pd.to_numeric(df_ev["mask_press"], errors="coerce")

    # Filter to events with valid pressure
    df_ev_p = df_ev.dropna(subset=["mask_press"])
    df_ev_p = df_ev_p[df_ev_p["mask_press"] > 0]

    print(f"=== Pressure at Events ({len(df_ev_p)} events with pressure data) ===\n")

    # Group by event type and source
    for source in ["eve", "custom"]:
        df_src = df_ev_p[df_ev_p["source"] == source]
        if len(df_src) == 0:
            continue
        print(f"--- {source.upper()} events ---")
        print(f"  {'Event Type':30s} {'Count':>6s} {'Median':>8s} {'P5':>8s} {'P95':>8s} {'Mean':>8s}")
        for etype, grp in df_src.groupby("label"):
            p = grp["mask_press"]
            print(f"  {etype:30s} {len(grp):6d} {p.median():8.1f} {p.quantile(0.05):8.1f} "
                  f"{p.quantile(0.95):8.1f} {p.mean():8.1f}")
        print()
else:
    print("No event-level data available. Run Section 12 batch first.")

In [ ]:
# ── §16b  Pressure-stratified event rates ────────────────────────────────────

if len(df_all_events) > 0 and len(df_nightly_v2) > 0:
    # Use STR MaskPress.50 per night as the pressure for that night
    df_n = df_nightly_v2.copy()
    if "str_MaskPress.50" in df_n.columns:
        press_col = "str_MaskPress.50"
    elif "MaskPress.50" in df_n.columns:
        press_col = "MaskPress.50"
    else:
        press_col = None

    if press_col is not None:
        df_n["press_bin"] = pd.cut(df_n[press_col].dropna(),
                                    bins=[0, 5, 6, 7, 8, 9, 10, 25],
                                    labels=["<5", "5-6", "6-7", "7-8", "8-9", "9-10", ">10"])

        print("=== Nightly Metrics by Median Pressure Bin ===\n")
        metrics = ["str_AHI", "custom_ahi", "fl_mean", "PCI_6bpm", "ODI_3"]
        avail_m = [m for m in metrics if m in df_n.columns]

        grouped = df_n.groupby("press_bin")
        print(f"  {'Pressure':>10s} {'Nights':>7s}", end="")
        for m in avail_m:
            print(f" {m:>12s}", end="")
        print()

        for pbin, grp in grouped:
            print(f"  {str(pbin):>10s} {len(grp):7d}", end="")
            for m in avail_m:
                val = grp[m].median()
                print(f" {val:12.2f}" if not pd.isna(val) else f" {'—':>12s}", end="")
            print()
        print()

        # Plot: AHI by pressure bin
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        ax = axes[0]
        ahi_by_press = df_n.groupby("press_bin")["str_AHI"].median()
        ax.bar(range(len(ahi_by_press)), ahi_by_press.values, color="#d9534f", alpha=0.7)
        ax.set_xticks(range(len(ahi_by_press)))
        ax.set_xticklabels(ahi_by_press.index.astype(str), fontsize=8)
        ax.set_ylabel("Median AHI")
        ax.set_xlabel("Median Pressure (cmH2O)")
        ax.set_title("AHI by Pressure Bin")

        ax = axes[1]
        if "fl_mean" in df_n.columns:
            fl_by_press = df_n.groupby("press_bin")["fl_mean"].median()
            ax.bar(range(len(fl_by_press)), fl_by_press.values, color="#8e44ad", alpha=0.7)
            ax.set_xticks(range(len(fl_by_press)))
            ax.set_xticklabels(fl_by_press.index.astype(str), fontsize=8)
            ax.set_ylabel("Median FlowLim Mean")
            ax.set_xlabel("Median Pressure (cmH2O)")
            ax.set_title("Flow Limitation by Pressure Bin")

        plt.tight_layout()
        plt.show()
    else:
        print("No pressure column available for stratification.")
else:
    print("Run Section 12 batch first.")

---
## 17. Therapy Trajectory

Linear regression on AHI and CTQI over time. Per-settings-period slopes.

In [ ]:
# ── §17a  Therapy trajectory ─────────────────────────────────────────────────

traj_ahi = change_summary.get("trajectory_ahi", {})
traj_ctqi = change_summary.get("trajectory_ctqi", {})

print("=== Therapy Trajectory ===\n")

if traj_ahi:
    overall = traj_ahi.get("overall", {})
    if overall:
        slope_mo = overall.get("slope_per_month", 0)
        r2 = overall.get("r_squared", 0)
        print(f"AHI overall trend:")
        print(f"  Slope: {slope_mo:+.4f} events/hr per month")
        print(f"  R²:    {r2:.4f}")
        direction = "declining" if slope_mo < 0 else "increasing" if slope_mo > 0 else "flat"
        print(f"  Direction: {direction}")

    segments = traj_ahi.get("segments", [])
    if segments:
        print(f"\n  Per-settings-period AHI trends ({len(segments)} periods):")
        print(f"  {'Start':>12s} {'End':>12s} {'Nights':>7s} {'Slope/mo':>10s} {'R²':>8s}")
        for seg in segments:
            print(f"  {str(seg.get('start',''))[:10]:>12s} {str(seg.get('end',''))[:10]:>12s} "
                  f"{seg.get('n', 0):7d} {seg.get('slope_per_month', 0):+10.4f} "
                  f"{seg.get('r_squared', 0):8.4f}")

if traj_ctqi:
    overall = traj_ctqi.get("overall", {})
    if overall:
        slope_mo = overall.get("slope_per_month", 0)
        r2 = overall.get("r_squared", 0)
        print(f"\nCTQI overall trend:")
        print(f"  Slope: {slope_mo:+.4f} points per month")
        print(f"  R²:    {r2:.4f}")

In [ ]:
# ── §17b  Trajectory plots ──────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# AHI trajectory with trend line
ax = axes[0]
ax.scatter(df_active.index, df_active["AHI"], s=6, alpha=0.3, color="#d9534f")
if len(df_active) >= 7:
    roll = df_active["AHI"].rolling(14, min_periods=5).median()
    ax.plot(df_active.index, roll, lw=2, color="#d9534f", label="14-day median")

# Overall trend line
traj = change_summary.get("trajectory_ahi", {}).get("overall", {})
if traj.get("slope_per_day") is not None:
    x_days = (df_active.index - df_active.index[0]).days
    y_trend = traj.get("intercept", 0) + traj["slope_per_day"] * x_days
    ax.plot(df_active.index, y_trend, ls="--", lw=1, color="#333",
            label=f"Trend ({traj.get('slope_per_month', 0):+.3f}/mo)")

# Settings change markers
change_dates = [pd.Timestamp(cd).to_pydatetime()
                for cd in change_summary.get("change_dates", [])]
for cd in change_dates:
    ax.axvline(cd, color="#333", ls=":", alpha=0.4, lw=0.5)

ax.set_ylabel("AHI (events/hr)")
ax.set_title("AHI Therapy Trajectory")
ax.legend(fontsize=8)

# CTQI trajectory (from batch data)
ax = axes[1]
df_ctqi = df_nightly_v2.dropna(subset=["CTQI"]).copy()
if len(df_ctqi) > 0:
    df_ctqi["date_dt"] = pd.to_datetime(df_ctqi["date"])
    ax.scatter(df_ctqi["date_dt"], df_ctqi["CTQI"], s=6, alpha=0.3, color="#1a6a2a")
    if len(df_ctqi) >= 7:
        roll = df_ctqi.set_index("date_dt")["CTQI"].rolling(14, min_periods=5).median()
        ax.plot(roll.index, roll.values, lw=2, color="#1a6a2a", label="14-day median")
    for cd in change_dates:
        ax.axvline(cd, color="#333", ls=":", alpha=0.4, lw=0.5)
    ax.axhline(90, color="#1a8a1a", ls=":", alpha=0.4)
    ax.axhline(75, color="#b08a00", ls=":", alpha=0.4)
    ax.legend(fontsize=8)
ax.set_ylabel("CTQI (0-100)")
ax.set_title("CTQI Therapy Trajectory")

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Glasgow Index over time
if "glasgow_overall" in df_nightly_v2.columns:
    df_glas = df_nightly_v2.dropna(subset=["glasgow_overall"]).copy()
    df_glas["date_dt"] = pd.to_datetime(df_glas["date"])
    if len(df_glas) > 0:
        fig, ax = plt.subplots(figsize=(14, 3.5))
        ax.scatter(df_glas["date_dt"], df_glas["glasgow_overall"],
                   s=6, alpha=0.3, color="#8e44ad")
        if len(df_glas) >= 7:
            roll = df_glas.set_index("date_dt")["glasgow_overall"].rolling(
                14, min_periods=5).median()
            ax.plot(roll.index, roll.values, lw=2, color="#8e44ad",
                    label="14-day median")
        for cd in change_dates:
            ax.axvline(cd, color="#333", ls=":", alpha=0.4, lw=0.5)
        ax.set_ylabel("Glasgow Overall")
        ax.set_title("Glasgow Index Over Time (lower = more eupneic breathing)")
        ax.legend(fontsize=8)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()

        print(f"Glasgow Index: median={df_glas['glasgow_overall'].median():.3f}  "
              f"mean={df_glas['glasgow_overall'].mean():.3f}  "
              f"range={df_glas['glasgow_overall'].min():.3f} – "
              f"{df_glas['glasgow_overall'].max():.3f}  "
              f"({len(df_glas)} nights)")

---
## 18. Data Export Hub

Configurable CSV exports. Toggle each export on/off.
All files saved to the working directory.

In [ ]:
# ── §18a  Export configuration ───────────────────────────────────────────────

EXPORT_DIR = Path(".") / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

EXPORT_LAST_N_NIGHTS = None   # Set to e.g. 10 for testing, None for all

# Toggle each export
EXPORT_NIGHTLY_SUMMARY  = True    # nightly_summary_v2.csv — per-night metrics
EXPORT_EVENT_LIST       = False    # all_events.csv — every event with pressure context
EXPORT_EVE_EVENTS       = False    # eve_events.csv — ResMed scored events only
EXPORT_STR_RAW          = False    # str_raw.csv — full STR longitudinal data (always all)
EXPORT_COOCCURRENCE     = False    # cooccurrence.csv — pulse/EVE/desat co-occurrence
EXPORT_SETTINGS_CHANGES = False    # settings_changes.csv — detected settings changes (always all)
EXPORT_SLEEP_SCHEDULE   = False    # sleep_schedule.csv — bedtime/waketime/duration/weekday
EXPORT_GLASGOW_BREATHS  = False   # glasgow_breaths.csv — per-breath Glasgow components (slow)
EXPORT_PLD_RAW          = False   # pld/pld_YYYYMMDD.csv per night (slow, large)
EXPORT_SA2_RAW          = False   # sa2/sa2_YYYYMMDD.csv per night (slow)
EXPORT_BRP_RAW          = False   # brp/brp_YYYYMMDD.csv per night (slow, very large)

exported = []

In [ ]:
# ── §18b  Run exports ────────────────────────────────────────────────────────
import time as _time
import csv as _csv

_t_export = _time.perf_counter()

# ── Resolve night list for disk-reading exports ──
_all_folders = sorted(f for f in Path(DATALOG_ROOT).iterdir()
                      if f.is_dir() and re.match(r"\d{8}", f.name))
if EXPORT_LAST_N_NIGHTS is not None:
    _export_folders = _all_folders[-EXPORT_LAST_N_NIGHTS:]   # most recent N
    print(f"Export scope: last {EXPORT_LAST_N_NIGHTS} nights "
          f"(of {len(_all_folders)} total)")
else:
    _export_folders = _all_folders
    print(f"Export scope: all {len(_all_folders)} nights")
_export_dates = set(f.name for f in _export_folders)

# ── Filter in-memory DataFrames to match scope ──
_df_export = df_nightly_v2.copy()
_df_events_export = df_all_events.copy() if len(df_all_events) > 0 else pd.DataFrame()
if EXPORT_LAST_N_NIGHTS is not None:
    _df_export = _df_export[_df_export["date"].isin(_export_dates)]
    if len(_df_events_export) > 0:
        _df_events_export = _df_events_export[_df_events_export["date"].isin(_export_dates)]

# ── 1. Nightly summary ──
if EXPORT_NIGHTLY_SUMMARY and len(_df_export) > 0:
    p = EXPORT_DIR / "nightly_summary_v2.csv"
    _df_export.to_csv(p, index=False)
    exported.append(("nightly_summary_v2.csv", len(_df_export), "rows"))

# ── 2. All events (custom + EVE with pressure context) ──
if EXPORT_EVENT_LIST and len(_df_events_export) > 0:
    p = EXPORT_DIR / "all_events.csv"
    _df_events_export.to_csv(p, index=False)
    exported.append(("all_events.csv", len(_df_events_export), "events"))

# ── 3. EVE events only ──
if EXPORT_EVE_EVENTS and len(_df_events_export) > 0:
    df_eve_only = _df_events_export[_df_events_export["source"] == "eve"]
    if len(df_eve_only) > 0:
        p = EXPORT_DIR / "eve_events.csv"
        df_eve_only.to_csv(p, index=False)
        exported.append(("eve_events.csv", len(df_eve_only), "events"))

# ── 4. STR raw (always full — it's one file, not per-night) ──
if EXPORT_STR_RAW and df_str is not None:
    p = EXPORT_DIR / "str_raw.csv"
    df_str.to_csv(p)
    exported.append(("str_raw.csv", len(df_str), "days"))

# ── 5. Co-occurrence (always full — already computed) ──
if EXPORT_COOCCURRENCE:
    try:
        if len(df_cooc) > 0:
            p = EXPORT_DIR / "cooccurrence.csv"
            df_cooc.to_csv(p, index=False)
            exported.append(("cooccurrence.csv", len(df_cooc), "nights"))
    except NameError:
        print("  ⚠ df_cooc not found — run §14c first")

# ── 6. Settings changes (always full) ──
if EXPORT_SETTINGS_CHANGES:
    try:
        df_ch = change_summary.get("changes_df")
        if df_ch is not None and len(df_ch) > 0:
            p = EXPORT_DIR / "settings_changes.csv"
            df_ch.to_csv(p, index=False)
            exported.append(("settings_changes.csv", len(df_ch), "changes"))

        ba_rows = []
        for ba in change_summary.get("before_after", []):
            cd = str(ba["change_date"])
            for metric, comp in ba.get("comparisons", {}).items():
                if comp.get("insufficient"):
                    continue
                ba_rows.append({
                    "change_date": cd,
                    "confounded": ba.get("confounded", False),
                    "metric": metric,
                    "before_median": comp.get("before_median"),
                    "after_median": comp.get("after_median"),
                    "p_value": comp.get("p_value"),
                    "p_adjusted": comp.get("p_adjusted"),
                    "significant": comp.get("significant"),
                    "effect_size": comp.get("effect_size"),
                    "direction": comp.get("direction"),
                })
        if ba_rows:
            p = EXPORT_DIR / "before_after_comparisons.csv"
            pd.DataFrame(ba_rows).to_csv(p, index=False)
            exported.append(("before_after_comparisons.csv", len(ba_rows), "comparisons"))
    except NameError:
        print("  ⚠ change_summary not found — run §13 first")

# ── 7. Sleep schedule (always full — from STR) ──
if EXPORT_SLEEP_SCHEDULE:
    try:
        sched_cols = ["Duration", "AHI", "OAI", "CAI", "HI", "RIN",
                      "MaskOn_first", "MaskOff_last", "MaskPress.50", "MaskPress.95",
                      "Leak.50", "SpO2.50"]
        avail_s = [c for c in sched_cols if c in df_sched.columns]
        for extra in ["bedtime_norm", "wake_hour", "weekday", "is_weekend", "bedtime_clock"]:
            if extra in df_sched.columns:
                avail_s.append(extra)
        p = EXPORT_DIR / "sleep_schedule.csv"
        df_sched[avail_s].to_csv(p)
        exported.append(("sleep_schedule.csv", len(df_sched), "nights"))
    except NameError:
        print("  ⚠ df_sched not found — run §15 first")

# ── 8. Glasgow per-breath (re-reads BRP, slow) ──
if EXPORT_GLASGOW_BREATHS:
    p = EXPORT_DIR / "glasgow_breaths.csv"
    _glas_count = 0
    _glas_header_written = False
    _glas_f = None
    print("  Exporting Glasgow per-breath data...")
    for folder in tqdm(_export_folders, desc="Glasgow export"):
        try:
            sess = read_session(folder)
            sess_a = adapt_session(sess)
            if sess_a.get("brp") is None:
                continue
            det_i = event_detection.detect_all(sess_a)
            if det_i["session_too_short"]:
                continue
            breaths = det_i.get("breaths", [])
            if not breaths:
                continue
            dstr = folder.name
            for bi, b in enumerate(breaths):
                row = {"date": dstr, "breath_idx": bi}
                row.update({k: v for k, v in b.items()
                            if k not in ("insp_samples", "exp_samples")})
                if not _glas_header_written:
                    _glas_f = open(p, "w", newline="")
                    _glas_w = _csv.DictWriter(_glas_f, fieldnames=list(row.keys()))
                    _glas_w.writeheader()
                    _glas_header_written = True
                _glas_w.writerow(row)
                _glas_count += 1
        except Exception:
            continue
    if _glas_f is not None:
        _glas_f.close()
        exported.append(("glasgow_breaths.csv", _glas_count, "breaths"))

# ── 9. PLD raw per night (re-reads PLD, slow, large) ──
if EXPORT_PLD_RAW:
    pld_dir = EXPORT_DIR / "pld"
    pld_dir.mkdir(exist_ok=True)
    _pld_count = 0
    print("  Exporting PLD raw signals...")
    for folder in tqdm(_export_folders, desc="PLD export"):
        try:
            sess = read_session(folder)
            sess_a = adapt_session(sess)
            pld = sess_a.get("pld")
            if pld is None:
                continue
            dstr = folder.name
            fs = pld.get("fs", 0.5)
            start = pld.get("start_dt")
            pld_signals = ["MaskPress.2s", "Press.2s", "EprPress.2s", "Leak.2s",
                           "RespRate.2s", "TidVol.2s", "MinVent.2s", "Snore.2s", "FlowLim.2s"]
            avail_sigs = {s: pld[s] for s in pld_signals if s in pld and pld[s] is not None}
            if not avail_sigs:
                continue
            n_samples = len(next(iter(avail_sigs.values())))
            times = [start + timedelta(seconds=i/fs) for i in range(n_samples)] if start else range(n_samples)
            df_pld = pd.DataFrame(avail_sigs)
            df_pld.insert(0, "time", times)
            df_pld.to_csv(pld_dir / f"pld_{dstr}.csv", index=False)
            _pld_count += 1
        except Exception:
            continue
    exported.append((f"pld/ ({_pld_count} files)", _pld_count, "nights"))

# ── 10. SA2 raw per night (re-reads SA2, slow) ──
if EXPORT_SA2_RAW:
    sa2_dir = EXPORT_DIR / "sa2"
    sa2_dir.mkdir(exist_ok=True)
    _sa2_count = 0
    print("  Exporting SA2 raw signals...")
    for folder in tqdm(_export_folders, desc="SA2 export"):
        try:
            sess = read_session(folder)
            sa2 = sess.get("SA2")
            if sa2 is None:
                continue
            meta = sess.get("meta", {}).get("SA2", {})
            dstr = folder.name
            fs = meta.get("fs", {}).get("SpO2.1s", 1.0)
            start = meta.get("start_dt")
            spo2 = sa2.get("SpO2.1s")
            pulse = sa2.get("Pulse.1s")
            if spo2 is None and pulse is None:
                continue
            n_samples = len(spo2) if spo2 is not None else len(pulse)
            times = [start + timedelta(seconds=i/fs) for i in range(n_samples)] if start else range(n_samples)
            data = {"time": times}
            if spo2 is not None:
                data["SpO2"] = spo2
            if pulse is not None:
                data["Pulse"] = pulse
            pd.DataFrame(data).to_csv(sa2_dir / f"sa2_{dstr}.csv", index=False)
            _sa2_count += 1
        except Exception:
            continue
    exported.append((f"sa2/ ({_sa2_count} files)", _sa2_count, "nights"))

# ── 11. BRP raw per night (re-reads BRP, slow, very large) ──
if EXPORT_BRP_RAW:
    brp_dir = EXPORT_DIR / "brp"
    brp_dir.mkdir(exist_ok=True)
    _brp_count = 0
    print("  Exporting BRP raw signals...")
    for folder in tqdm(_export_folders, desc="BRP export"):
        try:
            sess = read_session(folder)
            sess_a = adapt_session(sess)
            brp = sess_a.get("brp")
            if brp is None:
                continue
            dstr = folder.name
            fs = brp.get("fs", 25.0)
            start = brp.get("start_dt")
            flow = brp.get("Flow.40ms")
            press = brp.get("Press.40ms")
            if flow is None:
                continue
            n_samples = len(flow)
            # Downsample timestamps: compute start + step, not per-sample datetime list
            # Write with sample index and seconds offset (much faster than datetime per row)
            data = {"seconds": np.arange(n_samples) / fs}
            data["Flow"] = flow
            if press is not None:
                data["Press"] = press
            df_brp = pd.DataFrame(data)
            df_brp.to_csv(brp_dir / f"brp_{dstr}.csv", index=False,
                          float_format="%.4f")
            _brp_count += 1
            # Write a small metadata sidecar
            with open(brp_dir / f"brp_{dstr}_meta.txt", "w") as mf:
                mf.write(f"start_dt={start}\nfs={fs}\nsamples={n_samples}\n")
        except Exception:
            continue
    exported.append((f"brp/ ({_brp_count} files)", _brp_count, "nights"))

# ── Summary ──
_export_elapsed = _time.perf_counter() - _t_export

print(f"\n=== Export Summary ({_export_elapsed:.1f}s) ===")
print(f"Export directory: {EXPORT_DIR.resolve()}")
print()
for fname, count, unit in exported:
    print(f"  ✓ {fname:40s} {count:>6,} {unit}")
if not exported:
    print("  No exports generated. Check flags above.")

---
## 19. Generate Reports

Produce self-contained HTML reports (single-night and longitudinal).

In [ ]:
# ── §19a  Single-night HTML report ───────────────────────────────────────────

DEVICE_SN = ""
if ID_JSON and Path(ID_JSON).exists():
    try:
        DEVICE_SN = _json.loads(Path(ID_JSON).read_text())["FlowGenerator"]["IdentificationProfiles"]["Product"]["SerialNumber"]
    except (KeyError, TypeError, OSError, ValueError):
        DEVICE_SN = ""
REPORT_DIR = Path(".")    # output to working directory

html_single = report.generate_single_night(
    date_str=INSPECT_DATE,
    session=session_adapted,
    detection_result=det,
    night_indices=night_idx,
    eve_comparison=comparison,
    str_row=str_row,
    device_sn=DEVICE_SN,
    output_dir=REPORT_DIR,
)
print(f"Single-night report: report_{INSPECT_DATE}.html  ({len(html_single):,} chars)")

In [ ]:
# ── §19b  Longitudinal HTML report ───────────────────────────────────────────

long_stats = indices.compute_longitudinal_stats(df_str)

# Build co-occurrence summary from §14c (if available)
cooc_summary = None
if 'df_cooc' in dir() and len(df_cooc) > 0:
    cooc_summary = {
        "n_nights":  len(df_cooc),
        "total_ps":  int(df_cooc["n_ps"].sum()),
        "total_eve": int(df_cooc["n_eve"].sum()),
        "total_ds":  int(df_cooc["n_ds"].sum()),
        "ps_w_eve":  int(df_cooc["ps_w_eve"].sum()),
        "ps_w_fl":   int(df_cooc["ps_w_fl"].sum()),
        "ps_w_ds":   int(df_cooc["ps_w_ds"].sum()),
        "eve_w_ps":  int(df_cooc["eve_w_ps"].sum()),
        "eve_w_fl":  int(df_cooc["eve_w_fl"].sum()),
        "eve_w_ds":  int(df_cooc["eve_w_ds"].sum()),
        "ds_w_eve":  int(df_cooc["ds_w_eve"].sum()),
        "ds_w_ps":   int(df_cooc["ds_w_ps"].sum()),
        "ds_w_fl":   int(df_cooc["ds_w_fl"].sum()),
    }
    print(f"Co-occurrence summary: {cooc_summary['n_nights']} nights, "
          f"{cooc_summary['total_ps']:,} pulse surges")
else:
    print("No co-occurrence data available (§14c not run or no SA2 nights)")

html_long = report.generate_longitudinal(
    df_str=df_str,
    df_nightly=df_nightly_v2,
    df_uars_diag=df_uars_diag_v2,   # NEW — for per-criterion UARS analysis
    change_summary=change_summary,
    longitudinal_stats=long_stats,
    cooc_summary=cooc_summary,
    device_sn=DEVICE_SN,
    output_dir=REPORT_DIR,
)
print(f"Longitudinal report: report_longitudinal.html  ({len(html_long):,} chars)")

In [ ]:
# ── §19c  Pipeline summary ───────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  CPAP Analytics Pipeline — Run Complete")
print("=" * 60)
print(f"  Nights processed:  {len(df_nightly_v2)}")
print(f"  Settings changes:  {change_summary['n_changes']}")
print(f"  Reports generated: report_{INSPECT_DATE}.html")
print(f"                     report_longitudinal.html")
print(f"  Nightly CSV:       nightly_summary_v2.csv")
if len(df_nightly_v2) > 0:
    ctqi_valid = df_nightly_v2["CTQI"].dropna()
    if len(ctqi_valid) > 0:
        print(f"  CTQI range:        {ctqi_valid.min():.1f} – {ctqi_valid.max():.1f} "
              f"(median {ctqi_valid.median():.1f})")
    uars_count = df_nightly_v2["uars_suggestive"].sum()
    print(f"  UARS-suggestive:   {int(uars_count)} nights "
          f"({100*uars_count/len(df_nightly_v2):.0f}%)")
print("=" * 60)

---
## Notes & Known Issues

| Item | Detail |
|------|--------|
| BRP sampling rate | 25 Hz (1500 samples / 60s record). Signal name `Flow.40ms` reflects 40ms period = 25 Hz. |
| SA2 zeros | Zero values in `SpO2.1s` and `Pulse.1s` = signal loss, not true zero. Use `filter_spo2_zeros()`. |
| Timezone | All timestamps are tz-naive local time throughout. Device timezone = UTC-5 (from `CurrentSettings.json`). |
| STR flush | STR.edf may not include today's session until the device next syncs. |
| EVE vs STR AHI | EVE gives per-event timestamps. STR AHI is ResMed's own nightly index — use STR as ground truth for AHI. |
| EDF file transfer | Always copy EDF files as binary. Any text-encoding step will corrupt signal data. |

---

## 20. CTQI Weight Sensitivity Analysis

Post-hoc analysis added after the main pipeline run to verify that the CTQI
composite score is not trivially dominated by any single domain.

The analysis re-derives the five CTQI sub-scores (AHI, SpO₂, leak, flow
limitation, PCI) from the stored nightly metrics using the same piecewise-linear
breakpoint tables and SpO₂ penalties as `indices.compute_ctqi`, then recombines
them under several alternative weighting schemes. Note: this uses STR-derived
SpO₂ and leak medians from the nightly summary, which differ slightly from the
signal-computed medians used by the original CTQI, so reconstructed `current`
scheme values are expected to track the stored `CTQI` closely but not exactly.
The analysis is about **shift across schemes**, not reproducing the exact
stored value.

Output: summary statistics per scheme, per-night CTQI range across schemes,
Spearman correlations between scheme rankings, and grade-band distributions.

In [ ]:
# ── §20a  CTQI sensitivity — setup and sub-score reconstruction ─────────────
#
# Reuses df_nightly_v2 if available in memory, otherwise loads the CSV.
# Reconstructs sub-scores inline (no changes to indices.py).

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

# Prefer in-memory DataFrame; fall back to CSV
try:
    df_sens = df_nightly_v2.copy()
    print(f"Using in-memory df_nightly_v2 ({len(df_sens)} rows)")
except NameError:
    csv_path = Path(".") / "nightly_summary_v2.csv"
    df_sens = pd.read_csv(csv_path)
    print(f"Loaded df_sens from {csv_path} ({len(df_sens)} rows)")

# Filter to valid nights (matches pipeline convention)
df_sens = df_sens[df_sens["duration_min"] >= 60].copy().reset_index(drop=True)
print(f"Valid nights (duration >= 60 min): {len(df_sens)}")

# Breakpoint tables — copied verbatim from indices.py (do not modify)
_AHI_BP  = [(0, 100), (1, 90), (5, 75), (10, 50), (20, 20), (30, 0)]
_SPO2_BP = [(88, 0), (90, 25), (92, 55), (94, 80), (96, 100)]
_LEAK_BP = [(0.05, 100), (0.10, 80), (0.20, 55), (0.40, 20), (0.60, 0)]
_FL_BP   = [(0.10, 100), (0.20, 80), (0.30, 55), (0.50, 20), (0.70, 0)]
_PCI_BP  = [(0, 100), (5, 80), (10, 55), (20, 20), (35, 0)]


def _pw(value, bp):
    """Piecewise-linear interpolation with clamping (matches indices._piecewise_linear)."""
    if pd.isna(value):
        return float("nan")
    if value <= bp[0][0]:
        return bp[0][1]
    if value >= bp[-1][0]:
        return bp[-1][1]
    for i in range(len(bp) - 1):
        x0, y0 = bp[i]
        x1, y1 = bp[i + 1]
        if x0 <= value <= x1:
            t = (value - x0) / (x1 - x0)
            return y0 + t * (y1 - y0)
    return bp[-1][1]


def _compute_subscores_tuple(row):
    """Re-derive the five CTQI sub-scores. Returns a tuple (not a Series)
    to avoid any pandas index-alignment issues across versions."""
    ahi = row.get("str_AHI")
    if pd.isna(ahi):
        ahi = row.get("custom_ahi")

    spo2 = row.get("str_SpO2.50")
    leak = row.get("str_Leak.50")
    fl   = row.get("fl_mean")
    pci  = row.get("PCI_6bpm")
    t90  = row.get("T90", 0.0)
    odi3 = row.get("ODI_3", 0.0)

    has_sa2 = pd.notna(spo2)

    s_ahi  = _pw(ahi,  _AHI_BP)
    s_spo2 = _pw(spo2, _SPO2_BP) if has_sa2 else np.nan
    s_leak = _pw(leak, _LEAK_BP)
    s_fl   = _pw(fl,   _FL_BP)
    s_pci  = _pw(pci,  _PCI_BP) if has_sa2 else np.nan

    if has_sa2 and not pd.isna(s_spo2):
        if pd.notna(t90) and t90 > 5.0:
            s_spo2 = max(0, s_spo2 - 10)
        if pd.notna(odi3) and odi3 > 10.0:
            s_spo2 = max(0, s_spo2 - 5)

    return (s_ahi, s_spo2, s_leak, s_fl, s_pci, has_sa2)


# Build sub-score columns by iterating rows and assigning directly —
# avoids the pandas.apply return-type ambiguity that caused KeyError.
_s_ahi  = []
_s_spo2 = []
_s_leak = []
_s_fl   = []
_s_pci  = []
_has_sa2 = []

for _, _row in df_sens.iterrows():
    a, b, c, d, e, f = _compute_subscores_tuple(_row)
    _s_ahi.append(a)
    _s_spo2.append(b)
    _s_leak.append(c)
    _s_fl.append(d)
    _s_pci.append(e)
    _has_sa2.append(f)

df_sens["s_ahi"]   = _s_ahi
df_sens["s_spo2"]  = _s_spo2
df_sens["s_leak"]  = _s_leak
df_sens["s_fl"]    = _s_fl
df_sens["s_pci"]   = _s_pci
df_sens["has_sa2"] = _has_sa2

print("Sub-scores computed.")
print(f"  Nights with oximetry (has_sa2): {df_sens['has_sa2'].sum()}")
print(f"  Nights without oximetry:        {(~df_sens['has_sa2']).sum()}")
print(f"\n  Sanity check — first row sub-scores:")
print(f"    s_ahi={df_sens['s_ahi'].iloc[0]}  s_spo2={df_sens['s_spo2'].iloc[0]}  "
      f"s_leak={df_sens['s_leak'].iloc[0]}  s_fl={df_sens['s_fl'].iloc[0]}  "
      f"s_pci={df_sens['s_pci'].iloc[0]}")

In [ ]:
# ── §20b  Compute CTQI under alternative weighting schemes ──────────────────

SCHEMES = {
    "current":      {"ahi": 0.25, "spo2": 0.25, "leak": 0.20, "fl": 0.20, "pci": 0.10},
    "ahi_dominant": {"ahi": 0.40, "spo2": 0.20, "leak": 0.15, "fl": 0.15, "pci": 0.10},
    "fl_dominant":  {"ahi": 0.15, "spo2": 0.20, "leak": 0.15, "fl": 0.40, "pci": 0.10},
    "equal":        {"ahi": 0.20, "spo2": 0.20, "leak": 0.20, "fl": 0.20, "pci": 0.20},
    "leak_heavy":   {"ahi": 0.20, "spo2": 0.20, "leak": 0.35, "fl": 0.15, "pci": 0.10},
}


def _ctqi_weighted(row, weights):
    """Compute CTQI under arbitrary weights, renormalizing over available components.
    
    Matches the missing-data handling in indices.compute_ctqi: components with
    NaN sub-scores are dropped, and the remaining weights are renormalized to
    sum to 1.
    """
    scores = {
        "ahi":  row["s_ahi"],
        "spo2": row["s_spo2"],
        "leak": row["s_leak"],
        "fl":   row["s_fl"],
        "pci":  row["s_pci"],
    }
    active = {k: w for k, w in weights.items()
              if not pd.isna(scores[k]) and w > 0}
    if len(active) == 0:
        return np.nan
    total_w = sum(active.values())
    ctqi = sum(scores[k] * (w / total_w) for k, w in active.items())
    return round(max(0.0, min(100.0, ctqi)), 1)


for _name, _w in SCHEMES.items():
    col = f"ctqi_{_name}"
    df_sens[col] = df_sens.apply(lambda r, w=_w: _ctqi_weighted(r, w), axis=1)

# ---- Sanity check: reconstructed 'current' vs stored CTQI ----
# Exact match not expected because stored CTQI uses signal-computed SpO2/leak
# medians rather than STR-derived ones. Target: median |diff| < 1 point.
cmp = df_sens[["CTQI", "ctqi_current"]].dropna()
cmp = cmp.assign(diff=cmp["ctqi_current"] - cmp["CTQI"])

print("Sanity check: reconstructed 'current' vs stored CTQI")
print(f"  n compared:            {len(cmp)}")
print(f"  median |diff|:         {cmp['diff'].abs().median():.2f}")
print(f"  mean diff (signed):    {cmp['diff'].mean():.3f}")
print(f"  max |diff|:            {cmp['diff'].abs().max():.2f}")
print(f"  nights |diff| < 1.0:   {(cmp['diff'].abs() < 1.0).sum()} / {len(cmp)} "
      f"({100*(cmp['diff'].abs() < 1.0).mean():.1f}%)")
print(f"  nights |diff| < 3.0:   {(cmp['diff'].abs() < 3.0).sum()} / {len(cmp)} "
      f"({100*(cmp['diff'].abs() < 3.0).mean():.1f}%)")

if cmp["diff"].abs().median() > 2.0:
    print("\n  WARNING: median discrepancy > 2.0 — review sub-score derivation")
else:
    print("\n  Reconstruction closely tracks stored CTQI — sensitivity comparison valid.")

In [ ]:
# ── §20c  Summary statistics per scheme ─────────────────────────────────────

_scheme_cols = [f"ctqi_{n}" for n in SCHEMES]

# Distribution summary
summary_rows = []
for name in SCHEMES:
    s = df_sens[f"ctqi_{name}"].dropna()
    summary_rows.append({
        "scheme":  name,
        "n":       len(s),
        "median":  round(s.median(), 2),
        "mean":    round(s.mean(), 2),
        "IQR_25":  round(s.quantile(0.25), 2),
        "IQR_75":  round(s.quantile(0.75), 2),
        "min":     round(s.min(), 2),
        "max":     round(s.max(), 2),
    })
df_sensitivity_summary = pd.DataFrame(summary_rows)
print("CTQI distribution under each weighting scheme")
print("=" * 78)
print(df_sensitivity_summary.to_string(index=False))

# Per-night range across all schemes
df_sens["ctqi_max_shift"] = (
    df_sens[_scheme_cols].max(axis=1) - df_sens[_scheme_cols].min(axis=1)
)
shifts = df_sens["ctqi_max_shift"].dropna()
print()
print("Per-night CTQI range across all five schemes")
print("=" * 78)
print(f"  median shift:           {shifts.median():.2f} points")
print(f"  mean shift:             {shifts.mean():.2f} points")
print(f"  IQR of shift:           [{shifts.quantile(0.25):.2f}, {shifts.quantile(0.75):.2f}]")
print(f"  max shift observed:     {shifts.max():.2f} points")
print(f"  nights with shift > 3:  {(shifts > 3).sum()} ({100*(shifts > 3).mean():.1f}%)")
print(f"  nights with shift > 5:  {(shifts > 5).sum()} ({100*(shifts > 5).mean():.1f}%)")
print(f"  nights with shift > 10: {(shifts > 10).sum()} ({100*(shifts > 10).mean():.1f}%)")

In [ ]:
# ── §20d  Spearman correlations and grade band distribution ─────────────────

# Spearman correlations between scheme rankings
print("Spearman correlation between scheme rankings")
print("=" * 78)
mat = df_sens[_scheme_cols].dropna()
print(f"(computed on {len(mat)} nights with all schemes valid)\n")
corrs = mat.corr(method="spearman")
print(corrs.round(3).to_string())


def _grade(x):
    if pd.isna(x):
        return "N/A"
    if x >= 90: return "Excellent"
    if x >= 75: return "Good"
    if x >= 55: return "Fair"
    if x >= 35: return "Poor"
    return "Very Poor"


bands = ["Excellent", "Good", "Fair", "Poor", "Very Poor"]
grade_rows = []
for name in SCHEMES:
    g = df_sens[f"ctqi_{name}"].dropna().apply(_grade)
    pcts = g.value_counts(normalize=True) * 100
    row = {"scheme": name}
    row.update({b: round(pcts.get(b, 0.0), 1) for b in bands})
    grade_rows.append(row)
df_sensitivity_grades = pd.DataFrame(grade_rows)[["scheme"] + bands]

print()
print("Grade band distribution per scheme (% of valid nights)")
print("=" * 78)
print(df_sensitivity_grades.to_string(index=False))

In [ ]:
# ── §20e  Interpretation and optional export ────────────────────────────────

print("Interpretation summary")
print("=" * 78)

med_shift = df_sens["ctqi_max_shift"].median()
pct_shift_gt3 = 100 * (df_sens["ctqi_max_shift"] > 3).mean()
min_corr = corrs.where(~np.eye(len(corrs), dtype=bool)).stack().min()
max_median_diff = abs(df_sensitivity_summary["median"].max() -
                      df_sensitivity_summary["median"].min())

print(f"  Median per-night CTQI shift across schemes: {med_shift:.2f} points")
print(f"  Nights shifting >3 points across schemes:   {pct_shift_gt3:.1f}%")
print(f"  Minimum pairwise Spearman correlation:      {min_corr:.3f}")
print(f"  Range of median CTQI across schemes:        {max_median_diff:.2f} points")
print()

if min_corr > 0.95:
    print("  -> All weighting schemes produce highly correlated rankings (rho > 0.95).")
    print("     The relative ordering of nights is stable under reweighting; CTQI is")
    print("     not rigged to any single domain.")
if med_shift > 2:
    print(f"  -> Absolute CTQI values shift by a median of {med_shift:.1f} points across")
    print( "     schemes, which confirms each domain contributes meaningfully to the")
    print( "     composite.")
if pct_shift_gt3 > 10:
    print(f"  -> {pct_shift_gt3:.0f}% of nights shift by more than 3 points, meaning the weighting")
    print( "     choice is not cosmetic — it has real effect on absolute scores.")

# Optional: save summary tables for inclusion in report or appendix
SAVE_SENSITIVITY_CSV = False
if SAVE_SENSITIVITY_CSV:
    out_dir = Path(".") / "exports"
    out_dir.mkdir(exist_ok=True)

---

## 21. Validation Aggregation — All-Nights Custom-vs-EVE Comparison

Extends the single-night validation from §10d to the full dataset. For every
night processed by the batch loop in §12, `compare_with_eve` was computed at
both 15-second and 30-second tolerance windows. This section aggregates those
per-night results into dataset-wide precision, recall, and F1 distributions.

Nights with no EVE events produce NaN F1 and are excluded from the aggregate
medians. The original single-night result from §10d is located within the
distribution to show whether that night was representative.

In [ ]:
# ── §21a  Validation aggregation across all nights ──────────────────────────

import numpy as np
import pandas as pd
from pathlib import Path

# Prefer in-memory DataFrame; fall back to CSV
try:
    df_val = df_validation_v2.copy()
    print(f"Using in-memory df_validation_v2 ({len(df_val)} rows)")
except NameError:
    val_csv = Path(".") / "validation_v2.csv"
    df_val = pd.read_csv(val_csv)
    print(f"Loaded df_val from {val_csv} ({len(df_val)} rows)")

n_nights_total = df_val["date"].nunique()
print(f"Total unique nights with validation data: {n_nights_total}")

for tol in (15.0, 30.0):
    sub = df_val[df_val["tolerance_s"] == tol]
    n_nights = len(sub)
    n_no_eve = (sub["eve_total"] == 0).sum()
    n_no_custom = (sub["custom_total"] == 0).sum()
    n_both_empty = ((sub["eve_total"] == 0) & (sub["custom_total"] == 0)).sum()

    print(f"\nTolerance ±{tol:.0f}s  —  {n_nights} nights")
    print(f"  Nights with no EVE events:       {n_no_eve} ({100*n_no_eve/n_nights:.1f}%)")
    print(f"  Nights with no custom events:    {n_no_custom} ({100*n_no_custom/n_nights:.1f}%)")
    print(f"  Nights with no events at all:    {n_both_empty}")
    print(f"  Nights with events in both:      "
          f"{((sub['eve_total']>0) & (sub['custom_total']>0)).sum()}")
    print(f"  Total custom events across all nights: {sub['custom_total'].sum()}")
    print(f"  Total EVE events across all nights:    {sub['eve_total'].sum()}")
    print(f"  Total matched events:                  {sub['matched'].sum()}")

In [ ]:
# ── §21b  Per-metric distribution across nights ─────────────────────────────

def _summarize(series, label):
    s = series.dropna()
    if len(s) == 0:
        return {"metric": label, "n": 0, "median": float("nan"),
                "mean": float("nan"), "IQR_25": float("nan"),
                "IQR_75": float("nan"), "min": float("nan"),
                "max": float("nan")}
    return {
        "metric":  label,
        "n":       len(s),
        "median":  round(s.median(), 3),
        "mean":    round(s.mean(), 3),
        "IQR_25":  round(s.quantile(0.25), 3),
        "IQR_75":  round(s.quantile(0.75), 3),
        "min":     round(s.min(), 3),
        "max":     round(s.max(), 3),
    }


for tol in (15.0, 30.0):
    sub = df_val[df_val["tolerance_s"] == tol]
    print(f"\nPer-night metric distribution at ±{tol:.0f}s tolerance")
    print("=" * 78)
    rows = [
        _summarize(sub["precision"], "precision"),
        _summarize(sub["recall"],    "recall"),
        _summarize(sub["f1"],        "f1"),
    ]
    print(pd.DataFrame(rows).to_string(index=False))

    # Pooled (micro-averaged) precision/recall/F1 across the whole dataset
    tp = sub["matched"].sum()
    fp = (sub["custom_total"] - sub["matched"]).sum()
    fn = (sub["eve_total"] - sub["matched"]).sum()
    pooled_p = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    pooled_r = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    pooled_f1 = (2 * pooled_p * pooled_r / (pooled_p + pooled_r)
                 if (pooled_p + pooled_r) > 0 else float("nan"))

    print(f"\nPooled (micro-averaged) across all {sub['custom_total'].sum()} custom "
          f"and {sub['eve_total'].sum()} EVE events:")
    print(f"  precision: {pooled_p:.3f}")
    print(f"  recall:    {pooled_r:.3f}")
    print(f"  f1:        {pooled_f1:.3f}")

In [ ]:
# ── §21c  Locate the single-night INSPECT_DATE result within the distribution

# Default to the INSPECT_DATE used earlier in the notebook; fall back gracefully.
try:
    inspect_dstr = str(INSPECT_DATE)
except NameError:
    inspect_dstr = None

if inspect_dstr is not None:
    for tol in (15.0, 30.0):
        sub = df_val[df_val["tolerance_s"] == tol].dropna(subset=["f1"])
        if len(sub) == 0:
            continue
        match = sub[sub["date"].astype(str) == inspect_dstr]
        if len(match) == 0:
            print(f"\nINSPECT_DATE {inspect_dstr} not found in ±{tol:.0f}s validation set")
            continue
        row = match.iloc[0]
        f1_val = row["f1"]
        # What percentile of the F1 distribution does this sit at?
        pct = 100 * (sub["f1"] < f1_val).mean()
        med = sub["f1"].median()
        print(f"\nINSPECT_DATE {inspect_dstr} at ±{tol:.0f}s tolerance:")
        print(f"  F1:         {f1_val:.3f}  (dataset median {med:.3f})")
        print(f"  Percentile: {pct:.1f}%  of nights with valid F1 have lower F1")
        print(f"  custom={row['custom_total']}  eve={row['eve_total']}  "
              f"matched={row['matched']}")
else:
    print("INSPECT_DATE not defined in this session; skipping percentile check.")

In [ ]:
# ── §21d  Interpretation summary ────────────────────────────────────────────

print("Interpretation summary")
print("=" * 78)

for tol in (15.0, 30.0):
    sub = df_val[df_val["tolerance_s"] == tol].dropna(subset=["f1"])
    if len(sub) == 0:
        print(f"\nNo valid F1 values at ±{tol:.0f}s tolerance.")
        continue

    med_f1 = sub["f1"].median()
    iqr_lo = sub["f1"].quantile(0.25)
    iqr_hi = sub["f1"].quantile(0.75)
    n_valid = len(sub)
    total_nights = df_val[df_val["tolerance_s"] == tol].shape[0]

    print(f"\nAt ±{tol:.0f}s tolerance, across {n_valid} nights with events in "
          f"both custom and EVE streams (out of {total_nights} total):")
    print(f"  median per-night F1:  {med_f1:.3f}")
    print(f"  IQR [25th, 75th]:     [{iqr_lo:.3f}, {iqr_hi:.3f}]")
    print(f"  F1 >= 0.50:           {(sub['f1'] >= 0.50).sum()} nights "
          f"({100*(sub['f1'] >= 0.50).mean():.1f}%)")
    print(f"  F1 >= 0.75:           {(sub['f1'] >= 0.75).sum()} nights "
          f"({100*(sub['f1'] >= 0.75).mean():.1f}%)")

---

## 22. UARS Criterion Diagnostic

The UARS-suggestive flag (≥3 of 5 criteria) did not trigger on any night in
this dataset. Rather than adjust thresholds blindly, this section decomposes
the assessment into its five inputs and five criterion outcomes per night,
so that per-criterion firing rates and input-value distributions can be
inspected directly.

Each night contributes a row with:
- Raw input values (RIN, fl_mean, OAI, pressure_slope, PCI)
- Boolean firing status for each of the five criteria
- Overall uars_score (0–5) and uars_suggestive flag

From this, the diagnostic shows which criteria are discriminating, which are
pinned high or low, and where alternative thresholds would change behavior.

In [ ]:
# ── §22a  UARS per-criterion firing rates ───────────────────────────────────

import numpy as np
import pandas as pd
from pathlib import Path

try:
    df_uars = df_uars_diag_v2.copy()
    print(f"Using in-memory df_uars_diag_v2 ({len(df_uars)} rows)")
except NameError:
    uars_csv = Path(".") / "uars_diagnostic_v2.csv"
    df_uars = pd.read_csv(uars_csv)
    print(f"Loaded df_uars from {uars_csv} ({len(df_uars)} rows)")

print(f"\nuars_score distribution across {len(df_uars)} nights:")
print(df_uars["uars_score"].value_counts().sort_index().to_string())
print(f"\nuars_suggestive fired on: {df_uars['uars_suggestive'].sum()} nights")

crit_cols = ["crit_rin_elevated", "crit_fl_elevated", "crit_oai_low",
             "crit_pressure_rising", "crit_pci_elevated"]

print(f"\nPer-criterion firing rates")
print("=" * 70)
rows = []
for col in crit_cols:
    n_fire = int(df_uars[col].sum())
    rows.append({
        "criterion":        col.replace("crit_", ""),
        "nights_firing":    n_fire,
        "total_nights":     len(df_uars),
        "firing_rate_pct":  round(100 * n_fire / len(df_uars), 1),
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ── §22b  Input value distributions vs thresholds ───────────────────────────

# Defaults from assess_uars_composite signature
THRESHOLDS = {
    "rin":              ("rin > ",              5.0,  "rin_elevated"),
    "fl_frac_elevated": ("fl_frac_elevated > ", 0.15, "fl_elevated"),
    "oai":              ("oai < ",              5.0,  "oai_low"),
    "pressure_slope":   ("pressure_slope > ",   0.5,  "pressure_rising"),
    "pci":              ("pci > ",              5.0,  "pci_elevated"),
}

print("Input value distribution per criterion (n = nights with non-NaN value)")
print("=" * 78)
rows = []
for col, (_op, thresh, _crit) in THRESHOLDS.items():
    s = df_uars[col].dropna()
    rows.append({
        "input":   col,
        "threshold": thresh,
        "n":       len(s),
        "median":  round(s.median(), 4) if len(s) else float("nan"),
        "p25":     round(s.quantile(0.25), 4) if len(s) else float("nan"),
        "p75":     round(s.quantile(0.75), 4) if len(s) else float("nan"),
        "p95":     round(s.quantile(0.95), 4) if len(s) else float("nan"),
        "min":     round(s.min(), 4) if len(s) else float("nan"),
        "max":     round(s.max(), 4) if len(s) else float("nan"),
    })
print(pd.DataFrame(rows).to_string(index=False))

print(f"\nKey gap vs. threshold (how far the dataset sits from each threshold):")
for col, (_op, thresh, _crit) in THRESHOLDS.items():
    s = df_uars[col].dropna()
    if len(s) == 0:
        continue
    if "<" in _op:
        # "criterion is met when value is BELOW threshold"
        gap = thresh - s.max()
        direction = "below"
    else:
        # "criterion is met when value is ABOVE threshold"
        gap = s.max() - thresh
        direction = "above"
    near_boundary = ((s - thresh).abs() < 0.1 * max(abs(thresh), 0.01)).sum()
    print(f"  {col:16s} threshold={thresh:>6.2f}   "
          f"dataset max={s.max():>8.4f}   "
          f"extreme={'-' if gap <= 0 else '+'}{abs(gap):.4f} {direction} threshold")

In [ ]:
# ── §22c  Threshold sensitivity: what if thresholds were relaxed? ───────────

SWEEPS = {
    "rin":              [0.5, 1.0, 2.0, 3.0, 5.0],
    "fl_frac_elevated": [0.05, 0.10, 0.15, 0.20, 0.30, 0.50],
    "oai":              [1.0, 2.0, 5.0, 10.0],
    "pressure_slope":   [0.05, 0.10, 0.25, 0.50],
    "pci":              [3.0, 5.0, 7.0, 10.0],
}

DIRECTIONS = {
    "rin": ">", "fl_frac_elevated": ">", "oai": "<", "pressure_slope": ">", "pci": ">",
}

print("Threshold sweep: fraction of nights that would fire each criterion")
print("=" * 78)
for col, alts in SWEEPS.items():
    direction = DIRECTIONS[col]
    s = df_uars[col].dropna()
    if len(s) == 0:
        continue
    print(f"\n  {col}  (fires when value {direction} threshold)")
    for t in alts:
        if direction == ">":
            n = (s > t).sum()
        else:
            n = (s < t).sum()
        pct = 100 * n / len(s)
        print(f"    threshold = {t:>6.3f}:  {n:>4}/{len(s)} nights "
              f"({pct:>5.1f}%)")

In [ ]:
# ── §22d  Alternative rule simulation ───────────────────────────────────────
#
# Given the thresholds above, simulate how changing the rule would affect
# the number of uars-suggestive nights, without modifying the production
# assess_uars_composite function.

def _simulate(df, thresholds, rule_min):
    """Return number of nights that would be uars-suggestive under the given thresholds."""
    crits = pd.DataFrame({
        "rin":  (df["rin"].notna()) & (df["rin"] > thresholds["rin"]),
        "fl":   (df["fl_frac_elevated"].notna()) & (df["fl_frac_elevated"] > thresholds["fl_frac_elevated"]),
        "oai":  (df["oai"].notna()) & (df["oai"] < thresholds["oai"]),
        "prs":  (df["pressure_slope"].notna()) & (df["pressure_slope"] > thresholds["pressure_slope"]),
        "pci":  (df["pci"].notna()) & (df["pci"] > thresholds["pci"]),
    })
    score = crits.sum(axis=1)
    return int((score >= rule_min).sum()), score

# Scenario A: current production thresholds + current rule (>=3)
thr_A = {"rin": 5.0, "fl_frac_elevated": 0.15, "oai": 5.0, "pressure_slope": 0.5, "pci": 5.0}
n_A, sc_A = _simulate(df_uars, thr_A, 3)

# Scenario B: thresholds calibrated to this user's dataset
# Choose thresholds at the 90th percentile of the dataset so only elevated
# nights fire each criterion
thr_B = {
    "rin":              max(0.1, df_uars["rin"].quantile(0.90)),
    "fl_frac_elevated": max(0.05, df_uars["fl_frac_elevated"].quantile(0.90)),
    "oai":              df_uars["oai"].quantile(0.50),
    "pressure_slope":   max(0.05, df_uars["pressure_slope"].abs().quantile(0.90)),
    "pci":              df_uars["pci"].quantile(0.75),
}
n_B, sc_B = _simulate(df_uars, thr_B, 3)

# Scenario C: same dataset-calibrated thresholds + relaxed rule (>=2)
n_C, sc_C = _simulate(df_uars, thr_B, 2)

def _fmt_thr(d):
    return {k: round(v, 4) if isinstance(v, float) else v for k, v in d.items()}

print("Alternative rule simulation")
print("=" * 78)
print(f"\nA. Production thresholds + rule (>=3 of 5):")
print(f"   thresholds: {_fmt_thr(thr_A)}")
print(f"   uars-suggestive nights: {n_A}/{len(df_uars)} ({100*n_A/len(df_uars):.1f}%)")
print(f"   score distribution: {sc_A.value_counts().sort_index().to_dict()}")

print(f"\nB. Dataset-relative thresholds (90th pct of inputs) + rule (>=3 of 5):")
print(f"   thresholds: {_fmt_thr(thr_B)}")
print(f"   uars-suggestive nights: {n_B}/{len(df_uars)} ({100*n_B/len(df_uars):.1f}%)")
print(f"   score distribution: {sc_B.value_counts().sort_index().to_dict()}")

print(f"\nC. Dataset-relative thresholds + relaxed rule (>=2 of 5):")
print(f"   uars-suggestive nights: {n_C}/{len(df_uars)} ({100*n_C/len(df_uars):.1f}%)")
print(f"   score distribution: {sc_C.value_counts().sort_index().to_dict()}")

In [ ]:
# ── §22e  Interpretation summary ────────────────────────────────────────────

print("Interpretation summary")
print("=" * 78)

# Which criteria never or always fire?
for col in crit_cols:
    n = int(df_uars[col].sum())
    pct = 100 * n / len(df_uars)
    crit_name = col.replace("crit_", "")
    if pct == 0:
        print(f"  [NEVER FIRES]  {crit_name:20s}  threshold is too strict for this dataset")
    elif pct >= 95:
        print(f"  [ALMOST ALWAYS] {crit_name:20s}  {pct:.1f}% — threshold is trivially met")
    elif pct < 10:
        print(f"  [RARELY]       {crit_name:20s}  {pct:.1f}% — may still be informative on tail nights")
    else:
        print(f"  [DISCRIMINATES] {crit_name:20s}  {pct:.1f}% — reasonable range")

# Max achievable score under production thresholds
max_score = int(df_uars["uars_score"].max())
frac_at_max = 100 * (df_uars["uars_score"] == max_score).mean()
print(f"\n  Highest observed uars_score: {max_score} "
      f"(on {frac_at_max:.1f}% of nights)")
print(f"  Nights reaching uars_score >= 3: {int((df_uars['uars_score'] >= 3).sum())} "
      f"({100*(df_uars['uars_score'] >= 3).mean():.1f}%)")

# Suggested interpretation language for the report
print(f"\n  For report Section 5.1:")
print(f"  The UARS-suggestive flag did not trigger on any night. Per-criterion")
print(f"  decomposition shows that rin_elevated, fl_elevated, and pressure_rising")
print(f"  have firing rates at or near 0%, while oai_low fires on nearly every")
print(f"  night (trivially, given the user's low overall event burden) and")
print(f"  pci_elevated discriminates meaningfully. This pattern indicates the")
print(f"  published thresholds are calibrated against overt UARS presentations")
print(f"  documented in the clinical literature rather than this user's")
print(f"  subclinical profile. The pulse-flow co-occurrence analysis in Section 4")
print(f"  captures the UARS pattern in this data more sensitively than the")
print(f"  composite flag.")

In [ ]:
# ── §22f  fl_frac_elevated threshold calibration check ──────────────────────
#
# Post-Option-A sanity check: the UARS fl_elevated criterion now uses
# fl_frac_elevated (fraction of session with FlowLim.2s > 0.3) against a
# 0.15 threshold. This cell shows the dataset distribution so the threshold
# can be sanity-checked and tuned if needed.

if "fl_frac_elevated" not in df_uars.columns:
    print("Column 'fl_frac_elevated' not present — rerun §12 after applying "
          "the UARS fixes, then rerun §22.")
else:
    s = df_uars["fl_frac_elevated"].dropna()
    if len(s) == 0:
        print("fl_frac_elevated column is entirely NaN — check detect_all output.")
    else:
        print("fl_frac_elevated distribution across nights")
        print("=" * 78)
        print(f"  n (non-NaN):       {len(s)}")
        print(f"  min:               {s.min():.4f}")
        print(f"  25th percentile:   {s.quantile(0.25):.4f}")
        print(f"  median:            {s.median():.4f}")
        print(f"  75th percentile:   {s.quantile(0.75):.4f}")
        print(f"  90th percentile:   {s.quantile(0.90):.4f}")
        print(f"  95th percentile:   {s.quantile(0.95):.4f}")
        print(f"  99th percentile:   {s.quantile(0.99):.4f}")
        print(f"  max:               {s.max():.4f}")

        print()
        print("Fraction of nights that would fire 'fl_elevated' at various thresholds")
        print("=" * 78)
        for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
            n = int((s > t).sum())
            pct = 100 * n / len(s)
            marker = "  <-- current production threshold" if abs(t - 0.15) < 1e-6 else ""
            print(f"  threshold = {t:>5.2f}:  {n:>4}/{len(s)} nights ({pct:>5.1f}%){marker}")

        print()
        print("Interpretation guide")
        print("=" * 78)
        prod_fire_rate = 100 * (s > 0.15).mean()
        print(f"  At production threshold (0.15): {prod_fire_rate:.1f}% of nights fire fl_elevated.")

        if prod_fire_rate < 2:
            print(f"  -> VERY LOW firing rate. Threshold may be too strict for this user;")
            print(f"     fl_elevated will rarely contribute to the composite score.")
            print(f"     Consider lowering threshold — try the 90th percentile "
                  f"({s.quantile(0.90):.3f}) or 75th ({s.quantile(0.75):.3f}).")
        elif prod_fire_rate > 50:
            print(f"  -> VERY HIGH firing rate. Threshold is trivially met; criterion")
            print(f"     will contribute to score on most nights regardless of therapy")
            print(f"     quality. Consider raising threshold — try the 90th percentile "
                  f"({s.quantile(0.90):.3f}).")
        elif 5 <= prod_fire_rate <= 35:
            print(f"  -> Firing rate is in the discriminating range (5-35%). Threshold")
            print(f"     0.15 appears well-calibrated for this dataset.")
        else:
            print(f"  -> Firing rate is low-to-moderate. Threshold 0.15 is defensible but")
            print(f"     a lower value (e.g. 0.10) may increase discrimination if desired.")

In [ ]:
# ── §22g  One-line calibration verdict per criterion ───────────────────────

print("Per-criterion calibration verdict at production thresholds")
print("=" * 78)

PROD = [
    ("rin",              "rin",              5.0,  ">", 5.0),
    ("fl_frac_elevated", "fl_frac_elevated", 0.15, ">", 0.15),
    ("oai",              "oai",              5.0,  "<", 5.0),
    ("pressure_slope",   "pressure_slope",   0.5,  ">", 0.5),
    ("pci",              "pci",              5.0,  ">", 5.0),
]

for label, col, thr, op, _ in PROD:
    if col not in df_uars.columns:
        print(f"  {label:20s} COLUMN MISSING")
        continue
    s = df_uars[col].dropna()
    if len(s) == 0:
        print(f"  {label:20s} ALL NaN — input not populated")
        continue
    if op == ">":
        fire_rate = 100 * (s > thr).mean()
    else:
        fire_rate = 100 * (s < thr).mean()
    if fire_rate < 2:
        verdict = "RARELY fires — threshold too strict"
    elif fire_rate > 50:
        verdict = "FIRES MOSTLY — threshold trivially met"
    elif 5 <= fire_rate <= 35:
        verdict = "DISCRIMINATES — well-calibrated"
    else:
        verdict = "defensible but could tune"
    print(f"  {label:20s} n={len(s):>3}  fires {fire_rate:>5.1f}% — {verdict}")

In [ ]:
# ── §22h  UARS criterion analysis figure (3-panel) ─────────────────────────
#
# Replaces the binary UARS-suggestive flag plot. Three panels share the x-axis:
#   Panel 1 — uars_score over time (scatter + rolling median)
#   Panel 2 — per-criterion firing strip plot (tick per night that fires)
#   Panel 3 — per-criterion firing rate bar chart with discriminating band
#
# Use df_uars_diag_v2 (from §12) which contains the five crit_* booleans.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# Load diagnostic data
try:
    _dfu = df_uars_diag_v2.copy()
except NameError:
    _dfu = pd.read_csv(Path(".") / "uars_diagnostic_v2.csv")

_dfu["date_dt"] = pd.to_datetime(_dfu["date"], format="%Y%m%d")
_dfu = _dfu.sort_values("date_dt").reset_index(drop=True)

# Criterion columns — ordered as they appear in the composite rule
CRIT_ORDER = [
    ("crit_rin_elevated",    "RIN elevated\n(RIN > 5.0 /hr)"),
    ("crit_fl_elevated",     "FL elevated\n(fl_frac > 0.15)"),
    ("crit_oai_low",         "OAI low\n(OAI < 5.0 /hr)"),
    ("crit_pressure_rising", "Pressure rising\n(slope > 0.5 cmH₂O/hr)"),
    ("crit_pci_elevated",    "PCI elevated\n(PCI > 5.0 /hr)"),
]

fig = plt.figure(figsize=(12, 9))
gs = fig.add_gridspec(3, 1, height_ratios=[2.0, 2.5, 2.0], hspace=0.45)

# =========================================================================
# Panel 1 — uars_score over time
# =========================================================================
ax1 = fig.add_subplot(gs[0])
dates = _dfu["date_dt"]
scores = _dfu["uars_score"]

# Jitter scores slightly for visibility at integer y-values
np.random.seed(42)
jitter = np.random.uniform(-0.12, 0.12, size=len(scores))

ax1.scatter(dates, scores + jitter, s=10, alpha=0.35, color="#2C6F8F",
            edgecolors="none", label="Nightly score")

# Rolling median
if len(scores) >= 30:
    roll = scores.rolling(30, min_periods=10).median()
    ax1.plot(dates, roll, linewidth=2, color="#C04000", alpha=0.9,
             label="30-night rolling median")

ax1.axhline(3, color="#888888", linestyle="--", linewidth=0.8, alpha=0.6,
            zorder=1)
ax1.text(dates.iloc[-1], 3.05, " ≥3 of 5 threshold", fontsize=8,
         color="#666666", va="bottom", ha="right")

ax1.set_ylim(-0.5, 5.5)
ax1.set_yticks(range(0, 6))
ax1.set_ylabel("UARS score (0–5)")
ax1.set_title("Panel 1 — UARS composite score over time "
              "(sum of criteria met per night)", fontsize=10, loc="left")
ax1.legend(loc="upper left", fontsize=8)
ax1.grid(True, alpha=0.2)

# =========================================================================
# Panel 2 — per-criterion strip plot
# =========================================================================
ax2 = fig.add_subplot(gs[1], sharex=ax1)

# Row colors: red for never-fires, green for discriminating, blue for
# fires-trivially. Matches the firing-rate colors in panel 3.
for row_i, (col, label) in enumerate(CRIT_ORDER):
    fired_dates = dates[_dfu[col] == True]
    fire_rate = 100 * _dfu[col].mean()

    # Color code based on fire rate
    if fire_rate < 2:
        color = "#B0000A"  # deep red — criterion essentially dead
    elif fire_rate > 80:
        color = "#B0000A"  # same — trivially fires
    elif 5 <= fire_rate <= 50:
        color = "#2E7D32"  # green — discriminating
    else:
        color = "#B0000A"

    if len(fired_dates) > 0:
        ax2.scatter(fired_dates, np.full(len(fired_dates), row_i),
                    s=10, marker="|", color=color, alpha=0.55, linewidths=1.2)

    # Fire-rate annotation on the right
    ax2.text(1.005, row_i, f"{fire_rate:>5.1f}%", transform=ax2.get_yaxis_transform(),
             va="center", ha="left", fontsize=8, color="#333333",
             family="monospace")

ax2.set_yticks(range(len(CRIT_ORDER)))
ax2.set_yticklabels([label for _, label in CRIT_ORDER], fontsize=8)
ax2.set_ylim(-0.6, len(CRIT_ORDER) - 0.4)
ax2.invert_yaxis()
ax2.set_title("Panel 2 — Per-criterion firing strip "
              "(each tick = one night firing that criterion)",
              fontsize=10, loc="left")
ax2.grid(True, axis="x", alpha=0.2)

# =========================================================================
# Panel 3 — firing rate bar chart
# =========================================================================
ax3 = fig.add_subplot(gs[2])

fire_rates = [100 * _dfu[col].mean() for col, _ in CRIT_ORDER]
labels_short = ["rin", "fl", "oai", "pressure", "pci"]

# Discriminating band shading
ax3.axvspan(5, 50, color="#90C695", alpha=0.18, zorder=0)

# Bar colors match panel 2
bar_colors = []
for fr in fire_rates:
    if 5 <= fr <= 50:
        bar_colors.append("#2E7D32")
    else:
        bar_colors.append("#B0000A")

y_pos = np.arange(len(labels_short))
ax3.barh(y_pos, fire_rates, color=bar_colors, alpha=0.8, edgecolor="#333",
         linewidth=0.5)

# Value labels at end of each bar
for i, fr in enumerate(fire_rates):
    offset = 1.5 if fr < 95 else -1.5
    ha = "left" if fr < 95 else "right"
    color = "#000" if fr < 95 else "#FFF"
    ax3.text(fr + offset, i, f"{fr:.1f}%", va="center", ha=ha,
             fontsize=9, color=color)

ax3.set_yticks(y_pos)
ax3.set_yticklabels(labels_short, fontsize=9)
ax3.set_xlabel("% of nights firing criterion")
ax3.set_xlim(0, 105)
ax3.invert_yaxis()
ax3.set_title("Panel 3 — Per-criterion firing rate "
              "(green band = 5%–50% discriminating range)",
              fontsize=10, loc="left")
ax3.grid(True, axis="x", alpha=0.2)

# Format shared x-axis on panels 1 & 2
for ax in (ax1, ax2):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

ax2.tick_params(axis="x", labelbottom=True)
for lbl in ax2.get_xticklabels():
    lbl.set_rotation(30)
    lbl.set_ha("right")
ax1.tick_params(axis="x", labelbottom=False)

fig.suptitle("UARS Criterion Analysis — 503 nights",
             fontsize=12, fontweight="bold", y=0.995)

plt.show()

# Print one-line summary
print()
print("Summary")
print("=" * 70)
print(f"  Max UARS score observed: {int(_dfu['uars_score'].max())} "
      f"(on {int((_dfu['uars_score'] == _dfu['uars_score'].max()).sum())} nights)")
print(f"  Median UARS score: {_dfu['uars_score'].median():.1f}")
print(f"  Criteria in discriminating range (5%–50%): "
      f"{sum(1 for fr in fire_rates if 5 <= fr <= 50)} of 5")

---

## 23. Data Coverage Diagnostic — Night Counts Across Analyses

Different sections of this notebook report different numbers of nights (503,
449, 438, 425, 420, 404, 379, 189, 180, 172, 119). These are not errors and
mostly stem from legitimate data-availability differences: whether the
oximeter was worn, whether the session was long enough, whether events
were detected at all. This section catalogs every count used in the analysis
and explains what filter each one represents, so the denominator differences
can be addressed directly in the final report.

Output:
- Summary table of every denominator used
- Filter funnel from 503 → 449 → oximeter-worn subset
- Discrepancy flags for count pairs that should match but don't

In [ ]:
# ── §23a  Summary table: every night-count denominator in use ───────────────
#
# Assembles a single table of every "n" value used across the notebook's
# analyses, with the filter that produces it and a short explanation.
# Does not recompute anything — pulls from existing dataframes in memory.

import pandas as pd
from pathlib import Path

# Helper to count folders if needed
try:
    n_folders = sum(1 for p in Path(DATALOG_ROOT).iterdir()
                    if p.is_dir() and re.match(r"\d{8}", p.name))
except Exception:
    n_folders = None

# Collect counts from the in-memory dataframes. Each entry:
#   (section, label, count, filter_applied, explanation)
rows = []

# 1. Total nights ------------------------------------------------------------
n_str = len(df_str) if 'df_str' in dir() and df_str is not None else None
n_nightly = len(df_nightly_v2) if 'df_nightly_v2' in dir() else None

rows.append(("§1", "STR rows",
             n_str, "no filter",
             "one row per calendar day the device logged anything"))
rows.append(("§12", "df_nightly_v2 (all)",
             n_nightly, "no filter",
             "one row per DATALOG folder; matches STR 1:1"))

# 2. Valid / analyzable nights ----------------------------------------------
if 'df_nightly_v2' in dir():
    n_valid_60 = int((df_nightly_v2["duration_min"] >= 60).sum())
    n_valid_30 = int((df_nightly_v2["duration_min"] >= 30).sum())
    n_zero_dur = int((df_nightly_v2["duration_min"] < 1).sum())
else:
    n_valid_60 = n_valid_30 = n_zero_dur = None

rows.append(("§20", "Valid nights (CTQI sensitivity)",
             n_valid_60, "duration_min >= 60",
             "CTQI sensitivity filter — used across §20, §21, §22"))
rows.append(("§10/§12", "Session too short (excluded by detect_all)",
             n_valid_30 - n_valid_60 if (n_valid_30 is not None and n_valid_60 is not None) else None,
             "30 <= duration_min < 60",
             "detect_all accepts 30+ min but §20/21 require 60+"))
rows.append(("§12", "Zero-duration / placeholder rows",
             n_zero_dur, "duration_min < 1",
             "STR rows for days with no real session (missed nights)"))

# 3. Custom-vs-EVE validation (§21) -----------------------------------------
if 'df_validation_v2' in dir() and len(df_validation_v2) > 0:
    val15 = df_validation_v2[df_validation_v2["tolerance_s"] == 15.0]
    n_precision = int(val15["custom_total"].gt(0).sum())
    n_recall = int(val15["eve_total"].gt(0).sum())
    n_both = int(((val15["custom_total"] > 0) & (val15["eve_total"] > 0)).sum())
    n_f1_15 = int(val15["f1"].notna().sum())
    val30 = df_validation_v2[df_validation_v2["tolerance_s"] == 30.0]
    n_f1_30 = int(val30["f1"].notna().sum())
else:
    n_precision = n_recall = n_both = n_f1_15 = n_f1_30 = None

rows.append(("§21", "Precision defined",
             n_precision, "custom_total > 0",
             "nights where custom detectors fired at least one event"))
rows.append(("§21", "Recall defined",
             n_recall, "eve_total > 0",
             "nights where EVE contained at least one scored event"))
rows.append(("§21", "Both custom and EVE > 0",
             n_both, "custom > 0 AND eve > 0",
             "nights eligible for matching analysis"))
rows.append(("§21", "F1 defined (±15s)",
             n_f1_15, "custom > 0 AND eve > 0 AND matched > 0",
             "F1 is NaN when matched=0 (prec+rec denominator is 0)"))
rows.append(("§21", "F1 defined (±30s)",
             n_f1_30, "same, wider match tolerance",
             "more matches at 30s than 15s; more F1s defined"))

# 4. Oximetry / SA2 availability --------------------------------------------
if 'df_nightly_v2' in dir():
    # has_sa2 from CTQI reconstruction: approximated by spo2 median present
    if "str_SpO2.50" in df_nightly_v2.columns:
        n_sa2_present = int(df_nightly_v2["str_SpO2.50"].notna().sum())
    else:
        n_sa2_present = None
else:
    n_sa2_present = None

n_cooc_sa2_file = int(df_cooc_raw.shape[0]) if 'df_cooc_raw' in dir() else None
n_cooc_worn = int(df_cooc.shape[0]) if 'df_cooc' in dir() else None
n_cooc_eve = int((df_cooc["n_eve"] > 0).sum()) if 'df_cooc' in dir() else None
n_cooc_ds = int((df_cooc["n_ds"] > 0).sum()) if 'df_cooc' in dir() else None

rows.append(("§20", "CTQI: has_sa2 = True",
             n_sa2_present, "STR SpO2.50 not NaN",
             "oximetry data recorded in STR summary (may be sparse)"))
rows.append(("§14c", "SA2 file present",
             n_cooc_sa2_file, "session has 'sa2' block",
             "SA2.edf exists; oximeter may or may not have been worn"))
rows.append(("§14c", "Oximeter actually worn",
             n_cooc_worn, "n_pulse_surges > 0 OR n_desats > 0",
             "at least one pulse surge or desat event detected"))
rows.append(("§14c", "Nights with EVE events",
             n_cooc_eve, "n_eve > 0 (within oximeter-worn subset)",
             "EVE file has at least one non-boilerplate event"))
rows.append(("§14c", "Nights with desats",
             n_cooc_ds, "n_ds > 0 (within oximeter-worn subset)",
             "at least one SpO2 desaturation detected"))

# 5. UARS diagnostic --------------------------------------------------------
if 'df_uars_diag_v2' in dir() and len(df_uars_diag_v2) > 0:
    n_uars_rows = len(df_uars_diag_v2)
    n_uars_score_nonzero = int((df_uars_diag_v2["uars_score"] > 0).sum())
    n_uars_rin_valid = int(df_uars_diag_v2["rin"].notna().sum())
    n_uars_oai_valid = int(df_uars_diag_v2["oai"].notna().sum())
else:
    n_uars_rows = n_uars_score_nonzero = n_uars_rin_valid = n_uars_oai_valid = None

rows.append(("§22", "UARS diagnostic rows",
             n_uars_rows, "no filter",
             "one row per DATALOG folder, includes placeholder nights"))
rows.append(("§22", "UARS rin input valid",
             n_uars_rin_valid, "rin not NaN",
             "nights where STR row lookup succeeded and RIN was recorded"))
rows.append(("§22", "UARS oai input valid",
             n_uars_oai_valid, "oai not NaN",
             "nights where STR row lookup succeeded and OAI was recorded"))

# 6. Correlation and Glasgow ------------------------------------------------
if 'df_nightly_v2' in dir():
    corr_cols = ["custom_ahi", "PCI_6bpm", "fl_mean", "snore_index", "ODI_3"]
    avail = [c for c in corr_cols if c in df_nightly_v2.columns]
    n_any_metric = int(df_nightly_v2.dropna(subset=avail, how="all").shape[0]) \
        if avail else None
    n_glasgow = int(df_nightly_v2["glasgow_overall"].notna().sum()) \
        if "glasgow_overall" in df_nightly_v2.columns else None
else:
    n_any_metric = n_glasgow = None

rows.append(("§14a", "Any metric present",
             n_any_metric, "at least one of ahi/pci/fl/snore/odi3 not NaN",
             "correlation denominator — permissive"))
rows.append(("§14", "Glasgow index computed",
             n_glasgow, "glasgow_overall not NaN",
             "requires successful breath segmentation"))

# Build and print table
df_coverage = pd.DataFrame(rows, columns=[
    "section", "label", "n", "filter", "explanation"
])

print("Night-count inventory across the pipeline")
print("=" * 100)
pd.set_option("display.max_colwidth", 60)
print(df_coverage.to_string(index=False))
pd.reset_option("display.max_colwidth")

In [ ]:
# ── §23b  Filter funnel: 503 → valid → oximeter-worn ────────────────────────
#
# Shows where nights are lost at each stage, so the relationship between
# counts is visible as a cascade rather than as isolated numbers.

def _pct(n, total):
    return f"({100*n/total:.1f}%)" if total and total > 0 else ""

lines = []
lines.append("Filter funnel for the longitudinal dataset")
lines.append("=" * 78)

if n_nightly is not None:
    total = n_nightly
    lines.append(f"  {total:>4}  DATALOG folders / STR rows (all calendar days logged)")

    if n_zero_dur is not None:
        after_zero = total - n_zero_dur
        lines.append(f"    ├─ excluded: {n_zero_dur:>3} placeholder rows (duration < 1 min)")
        lines.append(f"  {after_zero:>4}  nights with any recorded session {_pct(after_zero, total)}")

    if n_valid_60 is not None:
        short_excluded = after_zero - n_valid_60
        lines.append(f"    ├─ excluded: {short_excluded:>3} sessions < 60 min "
                     f"(§20/§21/§22 threshold)")
        lines.append(f"  {n_valid_60:>4}  valid nights for analysis "
                     f"{_pct(n_valid_60, total)}")

    # Two branches from the valid set: validation and oximetry
    if n_both is not None and n_valid_60 is not None:
        lines.append("")
        lines.append(f"  Branch A — Custom-vs-EVE validation (§21)")
        lines.append(f"    {n_valid_60:>4}  valid nights")
        lines.append(f"     - {n_valid_60 - n_precision:>3} with no custom events "
                     f"-> {n_precision} precision-eligible")
        lines.append(f"     - {n_valid_60 - n_recall:>3} with no EVE events "
                     f"-> {n_recall} recall-eligible")
        lines.append(f"    {n_both:>4}  with events in both streams")
        if n_f1_15 is not None and n_f1_30 is not None:
            lines.append(f"     - {n_both - n_f1_15:>3} with 0 matches at ±15s "
                         f"-> {n_f1_15} F1-defined at ±15s")
            lines.append(f"     - {n_both - n_f1_30:>3} with 0 matches at ±30s "
                         f"-> {n_f1_30} F1-defined at ±30s")

    if n_cooc_sa2_file is not None and n_cooc_worn is not None:
        lines.append("")
        lines.append(f"  Branch B — Oximetry / co-occurrence (§14c)")
        lines.append(f"    {n_cooc_sa2_file:>4}  nights with SA2 file present")
        not_worn = n_cooc_sa2_file - n_cooc_worn
        lines.append(f"     - {not_worn:>3} with no pulse/desat events "
                     f"(oximeter not on finger)")
        lines.append(f"    {n_cooc_worn:>4}  nights with oximeter actually worn "
                     f"{_pct(n_cooc_worn, n_cooc_sa2_file)}")

print("\n".join(lines))

In [ ]:
# ── §23c  Discrepancy checks: counts that should match but might not ────────
#
# Flags pairs of counts that a reader might reasonably expect to be equal.
# Some differences are legitimate data-availability artifacts; others
# indicate real pipeline inconsistencies worth investigating.

print("Discrepancy checks")
print("=" * 78)

flags = []

def _check(label, a_name, a_val, b_name, b_val, expected_relation,
           explanation_if_ok, flag_if_mismatch):
    """Compare two counts against an expected relation.

    expected_relation: one of "==", "<=", ">=".
    """
    if a_val is None or b_val is None:
        flags.append(("SKIP", label,
                      f"missing data: {a_name}={a_val}, {b_name}={b_val}"))
        return
    ok = False
    if expected_relation == "==":
        ok = a_val == b_val
    elif expected_relation == "<=":
        ok = a_val <= b_val
    elif expected_relation == ">=":
        ok = a_val >= b_val
    if ok:
        flags.append(("OK", label,
                      f"{a_name}={a_val} {expected_relation} {b_name}={b_val}"
                      f" — {explanation_if_ok}"))
    else:
        flags.append(("FLAG", label,
                      f"{a_name}={a_val} vs {b_name}={b_val} "
                      f"— {flag_if_mismatch}"))

# Check 1: STR rows == DATALOG folders == df_nightly_v2
if n_str is not None and n_nightly is not None:
    _check("STR == DATALOG",
           "STR rows", n_str, "df_nightly_v2", n_nightly, "==",
           "every DATALOG folder has a matching STR row",
           "mismatch: one source has rows the other doesn't")

# Check 2: n_f1_15 <= n_f1_30 (wider tolerance can only match more)
if n_f1_15 is not None and n_f1_30 is not None:
    _check("F1 tolerance monotonicity",
           "F1 n (±15s)", n_f1_15, "F1 n (±30s)", n_f1_30, "<=",
           "wider tolerance strictly admits more F1-defined nights",
           "suspicious: wider window produced fewer valid F1s")

# Check 3: n_both <= n_valid_60
if n_both is not None and n_valid_60 is not None:
    _check("Both-streams ⊆ valid",
           "n_both", n_both, "n_valid_60", n_valid_60, "<=",
           "validation subset cannot exceed the valid-night set",
           "validation subset exceeds valid set — check the 60-min filter")

# Check 4: oximeter worn <= SA2 file present
if n_cooc_worn is not None and n_cooc_sa2_file is not None:
    _check("Worn ⊆ SA2 file",
           "worn", n_cooc_worn, "SA2 file", n_cooc_sa2_file, "<=",
           "oximeter-worn subset cannot exceed SA2-file subset",
           "wearing detected without an SA2 file — impossible")

# Check 5: UARS rin valid == UARS oai valid (both from STR)
if n_uars_rin_valid is not None and n_uars_oai_valid is not None:
    _check("RIN vs OAI coverage",
           "rin valid", n_uars_rin_valid, "oai valid", n_uars_oai_valid, "==",
           "both come from the same STR row lookup, so counts should match",
           "mismatch: one STR column has gaps the other doesn't — check STR parsing")

# Check 6: After the UARS fix, rin_valid should be close to n_nightly
# (every STR row should have RIN; only truly missing STR rows should be NaN)
if n_uars_rin_valid is not None and n_nightly is not None:
    gap = n_nightly - n_uars_rin_valid
    if gap == 0:
        flags.append(("OK", "RIN column coverage",
                      f"all {n_nightly} nights have RIN from STR"))
    elif gap < 20:
        flags.append(("OK", "RIN column coverage",
                      f"{gap} of {n_nightly} nights missing RIN "
                      "— likely genuine STR gaps (device-off days)"))
    else:
        flags.append(("FLAG", "RIN column coverage",
                      f"{gap} of {n_nightly} nights missing RIN "
                      "— >20 gaps; check if the STR row lookup is failing"))

# Check 7: has_sa2 (CTQI) vs oximeter-worn (§14c) — these should NOT match
# has_sa2 uses STR SpO2.50 presence (permissive); oximeter-worn uses event
# detection (strict). has_sa2 >= worn always.
if n_sa2_present is not None and n_cooc_worn is not None:
    _check("has_sa2 >= oximeter worn",
           "has_sa2 (STR)", n_sa2_present,
           "oximeter worn (§14c)", n_cooc_worn, ">=",
           "STR-based SpO2 presence is permissive; event-based is stricter",
           "unexpected: §14c found more worn-oximeter nights than STR SpO2")

# Print results
ok_count = sum(1 for f in flags if f[0] == "OK")
flag_count = sum(1 for f in flags if f[0] == "FLAG")
skip_count = sum(1 for f in flags if f[0] == "SKIP")

for status, label, detail in flags:
    marker = {"OK": "  [OK]  ", "FLAG": "[FLAG] ", "SKIP": "[SKIP] "}[status]
    print(f"{marker} {label}")
    print(f"         {detail}")
    print()

print(f"Summary: {ok_count} OK, {flag_count} flagged, {skip_count} skipped")
if flag_count == 0:
    print("All count differences are consistent with legitimate data-availability filters.")

In [ ]:
# ── §23d  Draft paragraph for the final report ──────────────────────────────
#
# Generates a ready-to-paste paragraph explaining denominator variation
# that can go into Section 3 (Methods) or Section 4 (Results) of the paper.

paragraph_lines = []
paragraph_lines.append("SUGGESTED REPORT PARAGRAPH (Section 3 or 4)")
paragraph_lines.append("=" * 78)
paragraph_lines.append("")

if all(v is not None for v in [n_nightly, n_valid_60, n_both, n_f1_15,
                                n_cooc_sa2_file, n_cooc_worn]):
    para = (
        f"The dataset comprises {n_nightly} calendar days of device use "
        f"({n_str} STR rows matching {n_nightly} DATALOG folders one-to-one). "
        f"Night counts reported across the results vary by analysis because "
        f"different analyses require different data to be present. Of the "
        f"{n_nightly} total rows, {n_valid_60} nights had sessions of at least "
        f"60 minutes and form the primary analysis set (used in §20 CTQI "
        f"sensitivity, §21 validation aggregation, and §22 UARS diagnostics). "
        f"For custom-vs-EVE validation (§21), {n_both} nights had events in "
        f"both the custom and ResMed-scored streams; per-night F1 was defined "
        f"on {n_f1_15} of these at a ±15 s tolerance and {n_f1_30} at ±30 s, "
        f"with the remainder having zero temporal matches (F1 undefined when "
        f"matched = 0). For oximetry-dependent co-occurrence analyses (§14c), "
        f"{n_cooc_sa2_file} nights had an SA2 file present but the Nonin "
        f"oximeter was only actually worn on {n_cooc_worn} of those "
        f"(identified by the presence of at least one pulse surge or "
        f"desaturation event); the remaining "
        f"{n_cooc_sa2_file - n_cooc_worn} nights reflect the finger-probe "
        f"being off during sleep. These denominator differences are reported "
        f"transparently in each subsection rather than forced to a common "
        f"filter, since each analysis's valid-night criterion reflects a "
        f"genuine data-availability constraint specific to that signal or "
        f"comparison."
    )
    # Wrap to ~80 cols
    import textwrap
    paragraph_lines.extend(textwrap.wrap(para, width=78))
else:
    paragraph_lines.append("Insufficient data to generate the draft paragraph.")
    paragraph_lines.append("Ensure §12 and §14c have been run in this session.")

print("\n".join(paragraph_lines))

In [ ]:
# Quick check: are STR rows outside the DATALOG date range?
str_dates = set(df_str.index.date)
datalog_dates = set(
    datetime.strptime(p.name, "%Y%m%d").date()
    for p in Path(DATALOG_ROOT).iterdir()
    if p.is_dir() and re.match(r"\d{8}", p.name)
)
str_only = sorted(str_dates - datalog_dates)
dl_only = sorted(datalog_dates - str_dates)
print(f"Dates in STR but not DATALOG: {len(str_only)}")
if str_only:
    print(f"  Range: {str_only[0]} -> {str_only[-1]}")
    print(f"  First 5: {str_only[:5]}")
    print(f"  Last 5: {str_only[-5:]}")
print(f"Dates in DATALOG but not STR: {len(dl_only)}")
if dl_only:
    print(f"  First 5: {dl_only[:5]}")